In [1]:
!pip install pycocotools

In [1]:
import shutil

# Define your source (input) and destination (output) paths
src_folder = '/kaggle/input/models/mayankgupta005/iw-detector-variants/pytorch/default/1/outputs'
dst_folder = '/kaggle/working/outputs'

# Copy the entire directory structure
shutil.copytree(src_folder, dst_folder)


'/kaggle/working/outputs'

In [3]:
"""
OceanXRay -- Improving the internal-wave detector, measured honestly
=====================================================================
Trains several detector "recipes" on S1-IW-2023 (same 593/75/74 split as
before), each with 3 random seeds, and reports mAP on the held-out test
set as mean +/- spread, plus bootstrap confidence intervals. With only 74
test images (108 waves), a 1-2 point difference can be pure luck; this
script tells you which differences are real.

RECIPES (edit EXPERIMENTS below to add more):
  baseline     -- the old recipe's settings: Faster R-CNN v1, no
                  augmentation, images shrunk to 800 px (torchvision
                  default), fixed LR, last epoch kept. Re-run here so the
                  comparison is fair (same code, same GPU, same seeds).
  improved     -- same model, plus:
                    * flips + 90-degree rotations (waves can travel any way,
                      so every training image counts 8x) + light brightness
                      / contrast jitter
                    * full 1024 px resolution (no shrinking)
                    * warm-up + cosine learning-rate schedule
                    * keep the epoch that scores best on the VALIDATION set
  improved_v2  -- same as 'improved' with torchvision's Faster R-CNN v2
                  (newer, stronger training recipe of the same design)

Every run is also evaluated with test-time augmentation (TTA): predict on
4 flipped copies of each test image and merge -- reported as a separate
column, never mixed with the plain numbers.

HOW TO RUN (Kaggle, GPU on)
---------------------------
  MODE = "selftest"  -- 1-2 min on CPU, tiny synthetic data; checks the
                        augmentation, metric and full pipeline. Run first.
  MODE = "train"     -- works through RUN_PLAN, skipping runs already in
                        RESULTS_CSV. Stops starting new runs before
                        TIME_BUDGET_HOURS, so it is safe to re-launch in a
                        new Kaggle session: it continues where it stopped
                        (copy OUT_DIR between sessions, e.g. as a dataset).
  MODE = "report"    -- comparison table, bootstrap intervals, bar chart.

Outputs (OUT_DIR): results.csv, preds_<run>.parquet (test predictions),
ckpt_<run>.pth (best weights), report.txt, comparison_chart.png
"""

import os
import sys
import json
import math
import time
import random

import numpy as np
import pandas as pd
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

# =========================== EDIT THESE ===========================
MODE = "report"            # "selftest" -> "train" -> "report"

DATASET_ROOT = "/kaggle/input/datasets/mayankgupta005/s1-iw-2023-datasetocean-internal-wave-dataset"
ANN_DIR = f"{DATASET_ROOT}/Annotation/Annotation"
IMAGE_DIR = f"{DATASET_ROOT}/IW_Image/IW_Image"
OUT_DIR = "outputs/iw_improve"

TIME_BUDGET_HOURS = 11.0     # Kaggle sessions end at 12 h
BATCH_SIZE = 4
NUM_WORKERS = 2

# Published results to show on the chart, for the SAME dataset/split only.
# Fill these from the papers; leave out anything not measured on S1-IW-2023.
PUBLISHED = [
    ("Paper: rFaster R-CNN (their model)", 91.20, 58.80),
    ("Paper: Faster R-CNN (R50 + FPN)",    87.17, 55.10),
    ("Paper: YOLOv9",                       87.80, None),
    ("Paper: RetinaNet + FPN",              84.86, None),
    ("Paper: SSD",                          84.64, None),
    ("Paper: YOLOv3",                       71.50, None),
]
RECIPE_LABELS = {"baseline": "Ours: original training",
                 "improved": "Ours: improved training",
                 "improved_v2": "Ours: improved training, v2 model"}
IOU_THRS = np.round(np.linspace(0.5, 0.95, 10), 2)
# ==================================================================

EXPERIMENTS = {
    # baseline = the settings of train_iw_detector.py (SGD lr 0.005, momentum
    # 0.9, weight decay 5e-4, no warm-up, no clipping, last epoch kept).
    "baseline":    dict(arch="v1", augment=False, img_size=800,  epochs=20, lr=0.005,
                        schedule="constant", weight_decay=5e-4, clip=None, select="last"),
    "improved":    dict(arch="v1", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    "improved_v2": dict(arch="v2", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
}
# Order matters: the most informative comparison first, so a single
# Kaggle session already answers "does the new recipe beat the old one?"
RUN_PLAN = [("baseline", 0), ("improved", 0),
            ("baseline", 1), ("improved", 1),
            ("baseline", 2), ("improved", 2),
            ("improved_v2", 0), ("improved_v2", 1), ("improved_v2", 2)]

VAL_EVERY = 2                # epochs between validation checks
N_BOOTSTRAP = 2000
NUM_CLASSES = 2              # background + internal wave
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_CSV = os.path.join(OUT_DIR, "results.csv")


# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
class COCODetectionDataset(Dataset):
    """Same loader as train_iw_detector.py, plus optional augmentation."""

    def __init__(self, annotations_path, image_dir, augment=False):
        with open(annotations_path) as f:
            coco = json.load(f)
        self.image_dir = image_dir
        self.augment = augment
        self.images = {img["id"]: img for img in coco["images"]}
        self.anns = {}
        for a in coco["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        self.image_ids = list(self.images.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.images[image_id]
        img = TF.to_tensor(Image.open(os.path.join(self.image_dir, info["file_name"])).convert("RGB"))
        boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]]
                 for a in self.anns.get(image_id, [])]
        labels = [a["category_id"] for a in self.anns.get(image_id, [])]
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        if self.augment:
            img, boxes = augment(img, boxes)
        return img, {"boxes": boxes,
                     "labels": torch.tensor(labels, dtype=torch.int64),
                     "image_id": torch.tensor([image_id])}


def augment(img, boxes):
    """All 8 flips/90-degree rotations (via transpose + flips, so box maths
    is exact) plus mild brightness/contrast jitter. boxes: [N,4] xyxy."""
    _, H, W = img.shape
    b = boxes.clone()
    if random.random() < 0.5:                      # transpose (swap x and y)
        img = img.transpose(1, 2)
        b = b[:, [1, 0, 3, 2]]
        H, W = W, H
    if random.random() < 0.5:                      # horizontal flip
        img = img.flip(-1)
        b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], dim=1)
    if random.random() < 0.5:                      # vertical flip
        img = img.flip(-2)
        b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], dim=1)
    if random.random() < 0.8:                      # brightness / contrast
        c = random.uniform(0.85, 1.15)
        br = random.uniform(-0.08, 0.08)
        m = img.mean()
        img = ((img - m) * c + m + br).clamp(0, 1)
    return img.contiguous(), b


def collate_fn(batch):
    return tuple(zip(*batch))


# ----------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------
def build_model(arch, img_size, pretrained=True):
    kw = dict(min_size=img_size, max_size=img_size)
    if arch == "v1":
        fn = torchvision.models.detection.fasterrcnn_resnet50_fpn
    elif arch == "v2":
        fn = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2
    else:
        raise ValueError(arch)
    model = fn(weights="DEFAULT", **kw) if pretrained else fn(weights=None, weights_backbone=None, **kw)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
    return model


def lr_at(epoch_float, cfg):
    if cfg["schedule"] == "constant":          # old recipe: fixed LR, no warm-up
        return cfg["lr"]
    warm = 1.0
    if epoch_float < warm:
        return cfg["lr"] * (0.1 + 0.9 * epoch_float / warm)
    t = (epoch_float - warm) / max(cfg["epochs"] - warm, 1e-6)
    return cfg["lr"] * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(t, 1.0))))


# ----------------------------------------------------------------------
# Inference (+ test-time augmentation)
# ----------------------------------------------------------------------
@torch.no_grad()
def predict(model, img, tta=False):
    """Returns (boxes [N,4] xyxy, scores [N]) on CPU."""
    _, H, W = img.shape
    outs = []
    variants = [(False, False)] + ([(True, False), (False, True), (True, True)] if tta else [])
    for hf, vf in variants:
        x = img
        if hf:
            x = x.flip(-1)
        if vf:
            x = x.flip(-2)
        o = model([x.to(DEVICE)])[0]
        b = o["boxes"].float().cpu()
        if hf:
            b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], 1)
        if vf:
            b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], 1)
        outs.append((b, o["scores"].float().cpu()))
    boxes = torch.cat([o[0] for o in outs])
    scores = torch.cat([o[1] for o in outs])
    if tta and len(boxes):
        boxes, scores = fuse_boxes(boxes, scores, n_views=len(variants))
    return boxes, scores


def fuse_boxes(boxes, scores, n_views, iou_thr=0.55):
    """Weighted box fusion (simple version): cluster overlapping boxes from
    the flipped views, average their coordinates weighted by score, and
    scale the score by how many views agreed."""
    order = scores.argsort(descending=True)
    boxes, scores = boxes[order], scores[order]
    used = torch.zeros(len(boxes), dtype=torch.bool)
    fused_b, fused_s = [], []
    iou = torchvision.ops.box_iou(boxes, boxes)
    for i in range(len(boxes)):
        if used[i]:
            continue
        grp = (~used) & (iou[i] >= iou_thr)
        grp[i] = True
        used |= grp
        w = scores[grp]
        fused_b.append((boxes[grp] * w[:, None]).sum(0) / w.sum())
        fused_s.append(w.mean() * min(int(grp.sum()), n_views) / n_views)
    return torch.stack(fused_b), torch.stack(fused_s)


def predict_split(model, ds, tta=False):
    model.eval()
    rows = []
    for idx in range(len(ds)):
        img, tgt = ds[idx]
        b, s = predict(model, img, tta=tta)
        iid = int(tgt["image_id"])
        for bb, ss in zip(b.tolist(), s.tolist()):
            rows.append(dict(image_id=iid, x1=bb[0], y1=bb[1], x2=bb[2], y2=bb[3], score=ss))
    return pd.DataFrame(rows, columns=["image_id", "x1", "y1", "x2", "y2", "score"])


# ----------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------
def coco_eval(ann_path, preds):
    """Official pycocotools numbers: returns (AP50, AP50:95) in %."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    if len(preds) == 0:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(ann_path)
        res = [dict(image_id=int(r.image_id), category_id=1,
                    bbox=[r.x1, r.y1, r.x2 - r.x1, r.y2 - r.y1], score=r.score)
               for r in preds.itertuples()]
        dt = gt.loadRes(res)
        ev = COCOeval(gt, dt, iouType="bbox")
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return 100 * ev.stats[1], 100 * ev.stats[0]


def per_image_matches(ann_path, preds, iou_thr=0.5, max_dets=100):
    """Greedy COCO-style matching at one IoU. Returns {image_id: (scores,
    is_tp, n_gt)} so AP can be recomputed quickly on bootstrap samples."""
    with open(ann_path) as f:
        coco = json.load(f)
    gt = {}
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        gt.setdefault(a["image_id"], []).append([x, y, x + w, y + h])
    out = {}
    for img in coco["images"]:
        iid = img["id"]
        g = torch.tensor(gt.get(iid, []), dtype=torch.float32).reshape(-1, 4)
        p = preds[preds.image_id == iid].sort_values("score", ascending=False).head(max_dets)
        sc = p.score.to_numpy()
        tp = np.zeros(len(p), dtype=bool)
        if len(p) and len(g):
            ious = torchvision.ops.box_iou(torch.tensor(p[["x1", "y1", "x2", "y2"]].to_numpy(),
                                                        dtype=torch.float32), g).numpy()
            taken = np.zeros(len(g), dtype=bool)
            for k in range(len(p)):
                cand = np.where(~taken & (ious[k] >= iou_thr))[0]
                if len(cand):
                    j = cand[np.argmax(ious[k, cand])]
                    taken[j] = True
                    tp[k] = True
        out[iid] = (sc, tp, len(g))
    return out


def ap_from_matches(matches, image_ids):
    """COCO 101-point interpolated AP for a (possibly repeated) list of images."""
    sc = np.concatenate([matches[i][0] for i in image_ids]) if len(image_ids) else np.array([])
    tp = np.concatenate([matches[i][1] for i in image_ids]) if len(image_ids) else np.array([])
    n_gt = sum(matches[i][2] for i in image_ids)
    if n_gt == 0:
        return np.nan
    order = np.argsort(-sc, kind="mergesort")
    tp = tp[order].astype(float)
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    for k in range(len(prec) - 2, -1, -1):          # make precision monotone
        prec[k] = max(prec[k], prec[k + 1])
    rs = np.linspace(0, 1, 101)
    idx = np.searchsorted(rec, rs, side="left")
    return 100 * np.mean([prec[i] if i < len(prec) else 0.0 for i in idx])


# ----------------------------------------------------------------------
# Training one run
# ----------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one(name, seed, ann_dir=ANN_DIR, image_dir=IMAGE_DIR, out_dir=OUT_DIR,
              pretrained=True, cfg_override=None, log_every=20):
    cfg = dict(EXPERIMENTS[name], **(cfg_override or {}))
    run_id = f"{name}_s{seed}"
    set_seed(seed)
    tr = COCODetectionDataset(f"{ann_dir}/instances_train.json", image_dir, augment=cfg["augment"])
    va = COCODetectionDataset(f"{ann_dir}/instances_val.json", image_dir)
    te = COCODetectionDataset(f"{ann_dir}/instances_test.json", image_dir)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, generator=g, drop_last=True)
    model = build_model(cfg["arch"], cfg["img_size"], pretrained).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=cfg["lr"], momentum=0.9, weight_decay=cfg["weight_decay"])
    use_amp = DEVICE.type == "cuda"     # mixed precision on GPU for speed (all recipes alike)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ckpt_path = os.path.join(out_dir, f"ckpt_{run_id}.pth")
    best_val, best_epoch, history = -1.0, -1, []
    t0 = time.time()
    steps_per_epoch = len(dl)
    for epoch in range(cfg["epochs"]):
        model.train()
        ep_loss = 0.0
        for step, (imgs, tgts) in enumerate(dl):
            for pg in opt.param_groups:
                pg["lr"] = lr_at(epoch + step / steps_per_epoch, cfg)
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                loss = sum(model(imgs, tgts).values())
            opt.zero_grad()
            scaler.scale(loss).backward()
            if cfg["clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update()
            ep_loss += loss.item()
            if step % log_every == 0:
                print(f"  [{run_id}] epoch {epoch + 1}/{cfg['epochs']} step {step}/{steps_per_epoch} "
                      f"loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.5f}")
        rec = dict(epoch=epoch + 1, loss=ep_loss / max(steps_per_epoch, 1),
                   minutes=(time.time() - t0) / 60)
        last = epoch == cfg["epochs"] - 1
        if cfg["select"] == "best_val" and ((epoch + 1) % VAL_EVERY == 0 or last):
            v50, _ = coco_eval(f"{ann_dir}/instances_val.json", predict_split(model, va))
            rec["val_ap50"] = v50
            if v50 > best_val:
                best_val, best_epoch = v50, epoch + 1
                torch.save(model.state_dict(), ckpt_path)
        if cfg["select"] == "last" and last:
            torch.save(model.state_dict(), ckpt_path)
            best_epoch = epoch + 1
        history.append(rec)
        print(f"  [{run_id}] epoch {epoch + 1} done: loss {rec['loss']:.4f}"
              + (f", val AP50 {rec['val_ap50']:.2f}" if "val_ap50" in rec else "")
              + f"  ({rec['minutes']:.1f} min so far)")
    pd.DataFrame(history).to_csv(os.path.join(out_dir, f"history_{run_id}.csv"), index=False)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_ann = f"{ann_dir}/instances_test.json"
    p_plain = predict_split(model, te, tta=False)
    p_tta = predict_split(model, te, tta=True)
    p_plain.to_parquet(os.path.join(out_dir, f"preds_{run_id}.parquet"), index=False)
    p_tta.to_parquet(os.path.join(out_dir, f"preds_{run_id}_tta.parquet"), index=False)
    ap50, ap = coco_eval(test_ann, p_plain)
    ap50_t, ap_t = coco_eval(test_ann, p_tta)
    row = dict(run_id=run_id, recipe=name, seed=seed, best_epoch=best_epoch,
               val_ap50=best_val if best_val >= 0 else np.nan,
               test_ap50=ap50, test_ap50_95=ap, test_ap50_tta=ap50_t, test_ap50_95_tta=ap_t,
               train_minutes=(time.time() - t0) / 60, **{f"cfg_{k}": v for k, v in cfg.items()})
    print(f"  [{run_id}] TEST  AP50 {ap50:.2f}  AP50:95 {ap:.2f}  |  with TTA: AP50 {ap50_t:.2f}  "
          f"AP50:95 {ap_t:.2f}")
    return row


def append_result(row, path=RESULTS_CSV):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def train_all():
    os.makedirs(OUT_DIR, exist_ok=True)
    done = set(pd.read_csv(RESULTS_CSV)["run_id"]) if os.path.exists(RESULTS_CSV) else set()
    start = time.time()
    durations = []
    for name, seed in RUN_PLAN:
        run_id = f"{name}_s{seed}"
        if run_id in done:
            print(f"skip {run_id} (already in results)")
            continue
        elapsed_h = (time.time() - start) / 3600
        est_h = (max(durations) if durations else 0) / 60
        if elapsed_h + est_h > TIME_BUDGET_HOURS:
            print(f"Stopping: {elapsed_h:.1f} h used, next run ~{est_h:.1f} h would exceed "
                  f"the {TIME_BUDGET_HOURS} h budget. Save {OUT_DIR} and re-run in a new session.")
            break
        print(f"\n=== {run_id} ===  config: {EXPERIMENTS[name]}")
        row = train_one(name, seed)
        append_result(row)
        durations.append(row["train_minutes"])
    print("\nDone for this session. Run MODE = 'report' for the comparison.")


# ----------------------------------------------------------------------
# Report
# ----------------------------------------------------------------------
def report(out_dir=OUT_DIR, ann_dir=ANN_DIR, make_chart=True):
    res = pd.read_csv(os.path.join(out_dir, "results.csv"))
    test_ann = f"{ann_dir}/instances_test.json"
    with open(test_ann) as f:
        img_ids = [im["id"] for im in json.load(f)["images"]]
    lines = []
    P = lambda s="": (print(s), lines.append(s))

    P("=" * 72)
    P("INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)")
    P("=" * 72)
    summary = (res.groupby("recipe")
               .agg(runs=("seed", "count"),
                    ap50_mean=("test_ap50", "mean"), ap50_sd=("test_ap50", "std"),
                    ap50_tta_mean=("test_ap50_tta", "mean"), ap50_tta_sd=("test_ap50_tta", "std"),
                    ap5095_mean=("test_ap50_95", "mean"), minutes=("train_minutes", "mean")))
    summary = summary.reindex([r for r in EXPERIMENTS if r in summary.index])
    P(f"{'recipe':13s} {'runs':>4s}  {'AP50 (mean ± sd)':>18s}  {'AP50 + TTA':>16s}  "
      f"{'AP50:95':>8s}  {'min/run':>7s}")
    for r, s in summary.iterrows():
        sd = lambda v: f"{v:.2f}" if pd.notna(v) else " n/a"
        P(f"{r:13s} {int(s.runs):4d}  {s.ap50_mean:9.2f} ± {sd(s.ap50_sd):>5s}  "
          f"{s.ap50_tta_mean:8.2f} ± {sd(s.ap50_tta_sd):>5s}  {s.ap5095_mean:8.2f}  {s.minutes:7.0f}")
    P()
    P("Individual runs:")
    P(res[["run_id", "best_epoch", "val_ap50", "test_ap50", "test_ap50_tta"]].round(2).to_string(index=False))

    # Bootstrap: resample test images; recompute every run's AP on the same sample
    rng = np.random.default_rng(0)
    matches = {}
    for run_id in res.run_id:
        for suffix in ("", "_tta"):
            p = os.path.join(out_dir, f"preds_{run_id}{suffix}.parquet")
            if os.path.exists(p):
                matches[run_id + suffix] = per_image_matches(test_ann, pd.read_parquet(p))
    first = res.sort_values("test_ap50").run_id.iloc[-1]
    check = ap_from_matches(matches[first], img_ids)
    P()
    P(f"Metric check: fast AP50 for {first} = {check:.2f} vs pycocotools "
      f"{res.loc[res.run_id == first, 'test_ap50'].iloc[0]:.2f} (should agree within ~0.5)")

    samples = [rng.choice(img_ids, size=len(img_ids), replace=True) for _ in range(N_BOOTSTRAP)]
    def recipe_ap(recipe, suffix, ids):
        runs = res[res.recipe == recipe].run_id
        return np.mean([ap_from_matches(matches[r + suffix], ids) for r in runs if r + suffix in matches])
    boot = {}
    for recipe in summary.index:
        for suffix in ("", "_tta"):
            boot[(recipe, suffix)] = np.array([recipe_ap(recipe, suffix, s) for s in samples])
    P()
    P("95% bootstrap intervals for each recipe (how much the number could move")
    P("with a different set of 74 test images):")
    for (recipe, suffix), v in boot.items():
        lo, hi = np.nanpercentile(v, [2.5, 97.5])
        P(f"  {recipe + (' + TTA' if suffix else ''):20s} {lo:6.2f} to {hi:6.2f}")
    if "baseline" in summary.index:
        P()
        P("Is each recipe really better than the old one? (paired bootstrap of the")
        P("difference vs baseline; 'yes' only if the whole 95% interval is above 0)")
        for (recipe, suffix), v in boot.items():
            if recipe == "baseline" and suffix == "":
                continue
            d = v - boot[("baseline", "")]
            lo, hi = np.nanpercentile(d, [2.5, 97.5])
            verdict = "yes" if lo > 0 else ("no -- worse" if hi < 0 else "not distinguishable")
            P(f"  {recipe + (' + TTA' if suffix else ''):20s} {np.nanmean(d):+6.2f} "
              f"(95%: {lo:+.2f} to {hi:+.2f})  -> {verdict}")
    for name, val in PUBLISHED.items():
        P()
        P(f"Published '{name}': {val:.2f} AP50. Recipes whose 95% interval contains it are")
        P("statistically tied with it on this test set; above it = genuinely better.")

    with open(os.path.join(out_dir, "report.txt"), "w") as f:
        f.write("\n".join(lines))

    if make_chart:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        labels, vals, errs, colors = [], [], [], []
        for recipe in summary.index:
            for suffix, tag in (("", ""), ("_tta", " + TTA")):
                v = boot[(recipe, suffix)]
                m = summary.loc[recipe, "ap50_tta_mean" if suffix else "ap50_mean"]
                lo, hi = np.nanpercentile(v, [2.5, 97.5])
                labels.append(f"Ours: {recipe}{tag}"); vals.append(m)
                errs.append([[m - lo], [hi - m]]); colors.append("#2a6fb0" if recipe != "baseline" else "#9aa7b4")
        for name, val in PUBLISHED.items():
            labels.append(name); vals.append(val); errs.append([[0], [0]]); colors.append("#d08a3a")
        fig, ax = plt.subplots(figsize=(9, 0.55 * len(labels) + 1.5))
        y = np.arange(len(labels))[::-1]
        ax.barh(y, vals, color=colors,
                xerr=np.hstack(errs) if errs else None, capsize=3)
        for yy, v in zip(y, vals):
            ax.text(v + 0.3, yy, f"{v:.1f}", va="center", fontsize=9)
        ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlim(max(0, min(vals) - 15), 100)
        ax.set_xlabel("mAP@0.5 on S1-IW-2023 test set (%)  -- bars: mean over seeds, lines: 95% bootstrap interval")
        ax.set_title("Internal-wave detection: same data, same test split")
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, "comparison_chart.png"), dpi=120)
        P(f"\nSaved {os.path.join(out_dir, 'comparison_chart.png')} and report.txt")


# ----------------------------------------------------------------------
# Self-test (CPU, synthetic, ~1-2 min)
# ----------------------------------------------------------------------
def selftest():
    import tempfile
    print("=" * 70 + "\nSELF-TEST\n" + "=" * 70)
    ok_all = True

    print("\n[1] Augmentation keeps boxes on the objects (all 8 orientations)")
    torch.manual_seed(0); random.seed(0)
    worst = 0.0
    for _ in range(64):
        img = torch.zeros(3, 120, 200)
        bx = torch.tensor([[30., 20., 90., 50.], [120., 60., 180., 110.]])
        for x1, y1, x2, y2 in bx.int().tolist():
            img[:, y1:y2, x1:x2] = 1.0
        a_img, a_b = augment(img, bx)
        m = a_img[0] > 0.5
        for x1, y1, x2, y2 in a_b.round().int().tolist():
            inside = m[y1:y2, x1:x2].float().mean().item()
            worst = max(worst, 1 - inside)
        if abs(m.sum().item() - sum(((b[2] - b[0]) * (b[3] - b[1])).item() for b in a_b)) > 1:
            worst = 1.0
    ok = worst < 0.01
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  worst box/object mismatch {worst:.3f}")

    print("\n[2] Fast AP50 matches pycocotools")
    tmp = tempfile.mkdtemp()
    rng = np.random.default_rng(1)
    images, anns, preds = [], [], []
    aid = 1
    for iid in range(1, 41):
        images.append(dict(id=iid, file_name=f"{iid}.png", width=256, height=256))
        for _ in range(rng.integers(1, 4)):
            x, y = rng.uniform(0, 180, 2); w, h = rng.uniform(20, 70, 2)
            anns.append(dict(id=aid, image_id=iid, category_id=1, bbox=[x, y, w, h], area=w * h, iscrowd=0))
            aid += 1
            if rng.random() < 0.8:
                j = rng.normal(0, 6, 4)
                preds.append(dict(image_id=iid, x1=x + j[0], y1=y + j[1], x2=x + w + j[2],
                                  y2=y + h + j[3], score=rng.uniform(0.3, 1)))
        for _ in range(rng.integers(0, 3)):
            x, y = rng.uniform(0, 200, 2)
            preds.append(dict(image_id=iid, x1=x, y1=y, x2=x + 40, y2=y + 40, score=rng.uniform(0, 0.8)))
    ann_path = os.path.join(tmp, "t.json")
    with open(ann_path, "w") as f:
        json.dump(dict(images=images, annotations=anns, categories=[dict(id=1, name="w")]), f)
    preds = pd.DataFrame(preds)
    ref, _ = coco_eval(ann_path, preds)
    fast = ap_from_matches(per_image_matches(ann_path, preds), [im["id"] for im in images])
    ok = abs(ref - fast) < 0.5
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  pycocotools {ref:.2f} vs fast {fast:.2f}")

    print("\n[3] Test-time augmentation maps flipped boxes back correctly")
    class Fake(torch.nn.Module):
        def forward(self, xs):                      # 'detects' the bright square
            outs = []
            for x in xs:
                ys, xs_ = torch.where(x[0] > 0.5)
                b = torch.tensor([[xs_.min(), ys.min(), xs_.max() + 1, ys.max() + 1]], dtype=torch.float32)
                outs.append(dict(boxes=b, scores=torch.tensor([0.9])))
            return outs
    img = torch.zeros(3, 100, 160); img[:, 10:30, 20:70] = 1
    b, s = predict(Fake().eval(), img, tta=True)
    ok = len(b) == 1 and torch.allclose(b[0], torch.tensor([20., 10., 70., 30.]), atol=0.5) and abs(s.item() - 0.9) < 1e-4
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  fused box {b.tolist()} score {s.tolist()}")

    print("\n[4] Faster R-CNN v2 builds and runs")
    try:
        m2 = build_model("v2", 192, pretrained=False).eval()
        with torch.no_grad():
            o = m2([torch.rand(3, 192, 192)])[0]
        ok = set(o) >= {"boxes", "scores", "labels"} and m2.roi_heads.box_predictor.cls_score.out_features == 2
    except Exception as e:
        print("  ", e); ok = False
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  v2 model forward pass, 2 output classes")

    print("\n[5] End-to-end: tiny synthetic dataset, 2 recipes x 2 seeds, train -> report (CPU)")
    img_dir = os.path.join(tmp, "img"); os.makedirs(img_dir)
    ann_dir = os.path.join(tmp, "ann"); os.makedirs(ann_dir)
    rng = np.random.default_rng(2)
    for split, n in (("train", 12), ("val", 4), ("test", 6)):
        ims, ans = [], []
        for k in range(n):
            iid = {"train": 0, "val": 100, "test": 200}[split] + k
            arr = (rng.normal(0.35, 0.08, (192, 192)) * 255).clip(0, 255).astype(np.uint8)
            x, y = rng.integers(10, 110, 2); w, h = rng.integers(40, 70, 2)
            for c in range(0, w, 12):                   # bright stripes = a 'wave packet'
                arr[y:y + h, x + c:x + c + 5] = 230
            Image.fromarray(arr).save(os.path.join(img_dir, f"{iid}.png"))
            ims.append(dict(id=iid, file_name=f"{iid}.png", width=192, height=192))
            ans.append(dict(id=iid + 1, image_id=iid, category_id=1, bbox=[int(x), int(y), int(w), int(h)],
                            area=int(w * h), iscrowd=0))
        with open(os.path.join(ann_dir, f"instances_{split}.json"), "w") as f:
            json.dump(dict(images=ims, annotations=ans, categories=[dict(id=1, name="internal wave")]), f)
    out = os.path.join(tmp, "out"); os.makedirs(out)
    global NUM_WORKERS, BATCH_SIZE
    nw, bs = NUM_WORKERS, BATCH_SIZE
    NUM_WORKERS, BATCH_SIZE = 0, 2
    try:
        for name, seed in [("baseline", 0), ("improved", 0), ("baseline", 1), ("improved", 1)]:
            row = train_one(name, seed, ann_dir=ann_dir, image_dir=img_dir, out_dir=out, pretrained=False,
                            cfg_override=dict(epochs=2, img_size=192), log_every=1000)
            append_result(row, os.path.join(out, "results.csv"))
        report(out_dir=out, ann_dir=ann_dir, make_chart=True)
        ok = os.path.exists(os.path.join(out, "comparison_chart.png"))
    except Exception as e:
        import traceback; traceback.print_exc()
        ok = False
    finally:
        NUM_WORKERS, BATCH_SIZE = nw, bs
    ok_all &= ok
    print(f"\n  {'PASS' if ok else 'FAIL'}  pipeline ran end to end (numbers above are meaningless --"
          f" untrained model, 2 epochs, synthetic data)")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- fix before training.")
    print("=" * 70)
    return ok_all


def main():
    in_nb = "ipykernel" in sys.modules
    mode = MODE
    if not in_nb and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "train", "report"):
        mode = sys.argv[1]
    print(f"Mode: {mode} | device: {DEVICE}")
    if mode == "selftest":
        selftest()
    elif mode == "train":
        train_all()
    elif mode == "report":
        report()
    else:
        print("MODE must be 'selftest', 'train' or 'report'")


if __name__ == "__main__":
    main()

Mode: report | device: cuda
INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)
recipe        runs    AP50 (mean ± sd)        AP50 + TTA   AP50:95  min/run
baseline         3      85.02 ±  1.43     88.63 ±  1.42     57.75       23
improved         3      91.08 ±  0.66     90.38 ±  0.35     61.26       43
improved_v2      3      92.27 ±  1.13     91.94 ±  1.35     64.38       49

Individual runs:
        run_id  best_epoch  val_ap50  test_ap50  test_ap50_tta
   baseline_s0          20       NaN      85.33          88.90
   improved_s0          20     90.52      90.32          90.28
   baseline_s1          20       NaN      86.27          89.90
   improved_s1          20     89.43      91.55          90.77
   baseline_s2          20       NaN      83.46          87.10
   improved_s2          20     89.21      91.37          90.08
improved_v2_s0          18     87.22      93.57          92.97
improved_v2_s1          22     87.49      91.71          90.41
improved_v2_s2     

AttributeError: 'list' object has no attribute 'items'

In [2]:
"""
OceanXRay -- Improving the internal-wave detector, measured honestly
=====================================================================
Trains several detector "recipes" on S1-IW-2023 (same 593/75/74 split as
before), each with 3 random seeds, and reports mAP on the held-out test
set as mean +/- spread, plus bootstrap confidence intervals. With only 74
test images (108 waves), a 1-2 point difference can be pure luck; this
script tells you which differences are real.

RECIPES (edit EXPERIMENTS below to add more):
  baseline     -- the old recipe's settings: Faster R-CNN v1, no
                  augmentation, images shrunk to 800 px (torchvision
                  default), fixed LR, last epoch kept. Re-run here so the
                  comparison is fair (same code, same GPU, same seeds).
  improved     -- same model, plus:
                    * flips + 90-degree rotations (waves can travel any way,
                      so every training image counts 8x) + light brightness
                      / contrast jitter
                    * full 1024 px resolution (no shrinking)
                    * warm-up + cosine learning-rate schedule
                    * keep the epoch that scores best on the VALIDATION set
  improved_v2  -- same as 'improved' with torchvision's Faster R-CNN v2
                  (newer, stronger training recipe of the same design)

Every run is also evaluated with test-time augmentation (TTA): predict on
4 flipped copies of each test image and merge -- reported as a separate
column, never mixed with the plain numbers.

HOW TO RUN (Kaggle, GPU on)
---------------------------
  MODE = "selftest"  -- 1-2 min on CPU, tiny synthetic data; checks the
                        augmentation, metric and full pipeline. Run first.
  MODE = "train"     -- works through RUN_PLAN, skipping runs already in
                        RESULTS_CSV. Stops starting new runs before
                        TIME_BUDGET_HOURS, so it is safe to re-launch in a
                        new Kaggle session: it continues where it stopped
                        (copy OUT_DIR between sessions, e.g. as a dataset).
  MODE = "report"    -- comparison table, bootstrap intervals, bar chart.

Outputs (OUT_DIR): results.csv, preds_<run>.parquet (test predictions),
ckpt_<run>.pth (best weights), report.txt, comparison_chart.png
"""

import os
import sys
import json
import math
import time
import random

import numpy as np
import pandas as pd
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

# =========================== EDIT THESE ===========================
MODE = "report"            # "selftest" -> "train" -> "report"

DATASET_ROOT = "/kaggle/input/datasets/mayankgupta005/s1-iw-2023-datasetocean-internal-wave-dataset"
ANN_DIR = f"{DATASET_ROOT}/Annotation/Annotation"
IMAGE_DIR = f"{DATASET_ROOT}/IW_Image/IW_Image"
OUT_DIR = "outputs/iw_improve"

TIME_BUDGET_HOURS = 11.0     # Kaggle sessions end at 12 h
BATCH_SIZE = 4
NUM_WORKERS = 2

# Published results to show on the chart, for the SAME dataset/split only.
# Fill these from the papers; leave out anything not measured on S1-IW-2023.
# (name, mAP@0.5, mAP@0.5:0.95 or None if not reported) -- from the reference
# paper's Tables 2-4. Note: the paper gives its own model 91.20 (Tables 3, 4)
# but 90.57 (Table 2) with otherwise identical numbers; 91.20 is used here.
PUBLISHED = [
    ("Paper: rFaster R-CNN (their model)", 91.20, 58.80),
    ("Paper: Faster R-CNN (R50 + FPN)",    87.17, 55.10),
    ("Paper: YOLOv9",                       87.80, None),
    ("Paper: RetinaNet + FPN",              84.86, None),
    ("Paper: SSD",                          84.64, None),
    ("Paper: YOLOv3",                       71.50, None),
]
RECIPE_LABELS = {"baseline": "Ours: original training",
                 "improved": "Ours: improved training",
                 "improved_v2": "Ours: improved training, v2 model"}
IOU_THRS = np.round(np.linspace(0.5, 0.95, 10), 2)
# ==================================================================

EXPERIMENTS = {
    # baseline = the settings of train_iw_detector.py (SGD lr 0.005, momentum
    # 0.9, weight decay 5e-4, no warm-up, no clipping, last epoch kept).
    "baseline":    dict(arch="v1", augment=False, img_size=800,  epochs=20, lr=0.005,
                        schedule="constant", weight_decay=5e-4, clip=None, select="last"),
    "improved":    dict(arch="v1", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    "improved_v2": dict(arch="v2", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
}
# Order matters: the most informative comparison first, so a single
# Kaggle session already answers "does the new recipe beat the old one?"
RUN_PLAN = [("baseline", 0), ("improved", 0),
            ("baseline", 1), ("improved", 1),
            ("baseline", 2), ("improved", 2),
            ("improved_v2", 0), ("improved_v2", 1), ("improved_v2", 2)]

VAL_EVERY = 2                # epochs between validation checks
N_BOOTSTRAP = 2000
NUM_CLASSES = 2              # background + internal wave
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_CSV = os.path.join(OUT_DIR, "results.csv")


# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
class COCODetectionDataset(Dataset):
    """Same loader as train_iw_detector.py, plus optional augmentation."""

    def __init__(self, annotations_path, image_dir, augment=False):
        with open(annotations_path) as f:
            coco = json.load(f)
        self.image_dir = image_dir
        self.augment = augment
        self.images = {img["id"]: img for img in coco["images"]}
        self.anns = {}
        for a in coco["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        self.image_ids = list(self.images.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.images[image_id]
        img = TF.to_tensor(Image.open(os.path.join(self.image_dir, info["file_name"])).convert("RGB"))
        boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]]
                 for a in self.anns.get(image_id, [])]
        labels = [a["category_id"] for a in self.anns.get(image_id, [])]
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        if self.augment:
            img, boxes = augment(img, boxes)
        return img, {"boxes": boxes,
                     "labels": torch.tensor(labels, dtype=torch.int64),
                     "image_id": torch.tensor([image_id])}


def augment(img, boxes):
    """All 8 flips/90-degree rotations (via transpose + flips, so box maths
    is exact) plus mild brightness/contrast jitter. boxes: [N,4] xyxy."""
    _, H, W = img.shape
    b = boxes.clone()
    if random.random() < 0.5:                      # transpose (swap x and y)
        img = img.transpose(1, 2)
        b = b[:, [1, 0, 3, 2]]
        H, W = W, H
    if random.random() < 0.5:                      # horizontal flip
        img = img.flip(-1)
        b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], dim=1)
    if random.random() < 0.5:                      # vertical flip
        img = img.flip(-2)
        b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], dim=1)
    if random.random() < 0.8:                      # brightness / contrast
        c = random.uniform(0.85, 1.15)
        br = random.uniform(-0.08, 0.08)
        m = img.mean()
        img = ((img - m) * c + m + br).clamp(0, 1)
    return img.contiguous(), b


def collate_fn(batch):
    return tuple(zip(*batch))


# ----------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------
def build_model(arch, img_size, pretrained=True):
    kw = dict(min_size=img_size, max_size=img_size)
    if arch == "v1":
        fn = torchvision.models.detection.fasterrcnn_resnet50_fpn
    elif arch == "v2":
        fn = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2
    else:
        raise ValueError(arch)
    model = fn(weights="DEFAULT", **kw) if pretrained else fn(weights=None, weights_backbone=None, **kw)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
    return model


def lr_at(epoch_float, cfg):
    if cfg["schedule"] == "constant":          # old recipe: fixed LR, no warm-up
        return cfg["lr"]
    warm = 1.0
    if epoch_float < warm:
        return cfg["lr"] * (0.1 + 0.9 * epoch_float / warm)
    t = (epoch_float - warm) / max(cfg["epochs"] - warm, 1e-6)
    return cfg["lr"] * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(t, 1.0))))


# ----------------------------------------------------------------------
# Inference (+ test-time augmentation)
# ----------------------------------------------------------------------
@torch.no_grad()
def predict(model, img, tta=False):
    """Returns (boxes [N,4] xyxy, scores [N]) on CPU."""
    _, H, W = img.shape
    outs = []
    variants = [(False, False)] + ([(True, False), (False, True), (True, True)] if tta else [])
    for hf, vf in variants:
        x = img
        if hf:
            x = x.flip(-1)
        if vf:
            x = x.flip(-2)
        o = model([x.to(DEVICE)])[0]
        b = o["boxes"].float().cpu()
        if hf:
            b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], 1)
        if vf:
            b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], 1)
        outs.append((b, o["scores"].float().cpu()))
    boxes = torch.cat([o[0] for o in outs])
    scores = torch.cat([o[1] for o in outs])
    if tta and len(boxes):
        boxes, scores = fuse_boxes(boxes, scores, n_views=len(variants))
    return boxes, scores


def fuse_boxes(boxes, scores, n_views, iou_thr=0.55):
    """Weighted box fusion (simple version): cluster overlapping boxes from
    the flipped views, average their coordinates weighted by score, and
    scale the score by how many views agreed."""
    order = scores.argsort(descending=True)
    boxes, scores = boxes[order], scores[order]
    used = torch.zeros(len(boxes), dtype=torch.bool)
    fused_b, fused_s = [], []
    iou = torchvision.ops.box_iou(boxes, boxes)
    for i in range(len(boxes)):
        if used[i]:
            continue
        grp = (~used) & (iou[i] >= iou_thr)
        grp[i] = True
        used |= grp
        w = scores[grp]
        fused_b.append((boxes[grp] * w[:, None]).sum(0) / w.sum())
        fused_s.append(w.mean() * min(int(grp.sum()), n_views) / n_views)
    return torch.stack(fused_b), torch.stack(fused_s)


def predict_split(model, ds, tta=False):
    model.eval()
    rows = []
    for idx in range(len(ds)):
        img, tgt = ds[idx]
        b, s = predict(model, img, tta=tta)
        iid = int(tgt["image_id"])
        for bb, ss in zip(b.tolist(), s.tolist()):
            rows.append(dict(image_id=iid, x1=bb[0], y1=bb[1], x2=bb[2], y2=bb[3], score=ss))
    return pd.DataFrame(rows, columns=["image_id", "x1", "y1", "x2", "y2", "score"])


# ----------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------
def coco_eval(ann_path, preds):
    """Official pycocotools numbers: returns (AP50, AP50:95) in %."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    if len(preds) == 0:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(ann_path)
        res = [dict(image_id=int(r.image_id), category_id=1,
                    bbox=[r.x1, r.y1, r.x2 - r.x1, r.y2 - r.y1], score=r.score)
               for r in preds.itertuples()]
        dt = gt.loadRes(res)
        ev = COCOeval(gt, dt, iouType="bbox")
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return 100 * ev.stats[1], 100 * ev.stats[0]


def per_image_matches(ann_path, preds, iou_thr=0.5, max_dets=100):
    """Greedy COCO-style matching at one IoU. Returns {image_id: (scores,
    is_tp, n_gt)} so AP can be recomputed quickly on bootstrap samples."""
    with open(ann_path) as f:
        coco = json.load(f)
    gt = {}
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        gt.setdefault(a["image_id"], []).append([x, y, x + w, y + h])
    out = {}
    for img in coco["images"]:
        iid = img["id"]
        g = torch.tensor(gt.get(iid, []), dtype=torch.float32).reshape(-1, 4)
        p = preds[preds.image_id == iid].sort_values("score", ascending=False).head(max_dets)
        sc = p.score.to_numpy()
        tp = np.zeros(len(p), dtype=bool)
        if len(p) and len(g):
            ious = torchvision.ops.box_iou(torch.tensor(p[["x1", "y1", "x2", "y2"]].to_numpy(),
                                                        dtype=torch.float32), g).numpy()
            taken = np.zeros(len(g), dtype=bool)
            for k in range(len(p)):
                cand = np.where(~taken & (ious[k] >= iou_thr))[0]
                if len(cand):
                    j = cand[np.argmax(ious[k, cand])]
                    taken[j] = True
                    tp[k] = True
        out[iid] = (sc, tp, len(g))
    return out


def ap_from_matches(matches, image_ids):
    """COCO 101-point interpolated AP for a (possibly repeated) list of images."""
    sc = np.concatenate([matches[i][0] for i in image_ids]) if len(image_ids) else np.array([])
    tp = np.concatenate([matches[i][1] for i in image_ids]) if len(image_ids) else np.array([])
    n_gt = sum(matches[i][2] for i in image_ids)
    if n_gt == 0:
        return np.nan
    order = np.argsort(-sc, kind="mergesort")
    tp = tp[order].astype(float)
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    for k in range(len(prec) - 2, -1, -1):          # make precision monotone
        prec[k] = max(prec[k], prec[k + 1])
    rs = np.linspace(0, 1, 101)
    idx = np.searchsorted(rec, rs, side="left")
    return 100 * np.mean([prec[i] if i < len(prec) else 0.0 for i in idx])


# ----------------------------------------------------------------------
# Training one run
# ----------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one(name, seed, ann_dir=ANN_DIR, image_dir=IMAGE_DIR, out_dir=OUT_DIR,
              pretrained=True, cfg_override=None, log_every=20):
    cfg = dict(EXPERIMENTS[name], **(cfg_override or {}))
    run_id = f"{name}_s{seed}"
    set_seed(seed)
    tr = COCODetectionDataset(f"{ann_dir}/instances_train.json", image_dir, augment=cfg["augment"])
    va = COCODetectionDataset(f"{ann_dir}/instances_val.json", image_dir)
    te = COCODetectionDataset(f"{ann_dir}/instances_test.json", image_dir)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, generator=g, drop_last=True)
    model = build_model(cfg["arch"], cfg["img_size"], pretrained).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=cfg["lr"], momentum=0.9, weight_decay=cfg["weight_decay"])
    use_amp = DEVICE.type == "cuda"     # mixed precision on GPU for speed (all recipes alike)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ckpt_path = os.path.join(out_dir, f"ckpt_{run_id}.pth")
    best_val, best_epoch, history = -1.0, -1, []
    t0 = time.time()
    steps_per_epoch = len(dl)
    for epoch in range(cfg["epochs"]):
        model.train()
        ep_loss = 0.0
        for step, (imgs, tgts) in enumerate(dl):
            for pg in opt.param_groups:
                pg["lr"] = lr_at(epoch + step / steps_per_epoch, cfg)
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                loss = sum(model(imgs, tgts).values())
            opt.zero_grad()
            scaler.scale(loss).backward()
            if cfg["clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update()
            ep_loss += loss.item()
            if step % log_every == 0:
                print(f"  [{run_id}] epoch {epoch + 1}/{cfg['epochs']} step {step}/{steps_per_epoch} "
                      f"loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.5f}")
        rec = dict(epoch=epoch + 1, loss=ep_loss / max(steps_per_epoch, 1),
                   minutes=(time.time() - t0) / 60)
        last = epoch == cfg["epochs"] - 1
        if cfg["select"] == "best_val" and ((epoch + 1) % VAL_EVERY == 0 or last):
            v50, _ = coco_eval(f"{ann_dir}/instances_val.json", predict_split(model, va))
            rec["val_ap50"] = v50
            if v50 > best_val:
                best_val, best_epoch = v50, epoch + 1
                torch.save(model.state_dict(), ckpt_path)
        if cfg["select"] == "last" and last:
            torch.save(model.state_dict(), ckpt_path)
            best_epoch = epoch + 1
        history.append(rec)
        print(f"  [{run_id}] epoch {epoch + 1} done: loss {rec['loss']:.4f}"
              + (f", val AP50 {rec['val_ap50']:.2f}" if "val_ap50" in rec else "")
              + f"  ({rec['minutes']:.1f} min so far)")
    pd.DataFrame(history).to_csv(os.path.join(out_dir, f"history_{run_id}.csv"), index=False)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_ann = f"{ann_dir}/instances_test.json"
    p_plain = predict_split(model, te, tta=False)
    p_tta = predict_split(model, te, tta=True)
    p_plain.to_parquet(os.path.join(out_dir, f"preds_{run_id}.parquet"), index=False)
    p_tta.to_parquet(os.path.join(out_dir, f"preds_{run_id}_tta.parquet"), index=False)
    ap50, ap = coco_eval(test_ann, p_plain)
    ap50_t, ap_t = coco_eval(test_ann, p_tta)
    row = dict(run_id=run_id, recipe=name, seed=seed, best_epoch=best_epoch,
               val_ap50=best_val if best_val >= 0 else np.nan,
               test_ap50=ap50, test_ap50_95=ap, test_ap50_tta=ap50_t, test_ap50_95_tta=ap_t,
               train_minutes=(time.time() - t0) / 60, **{f"cfg_{k}": v for k, v in cfg.items()})
    print(f"  [{run_id}] TEST  AP50 {ap50:.2f}  AP50:95 {ap:.2f}  |  with TTA: AP50 {ap50_t:.2f}  "
          f"AP50:95 {ap_t:.2f}")
    return row


def append_result(row, path=RESULTS_CSV):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def train_all():
    os.makedirs(OUT_DIR, exist_ok=True)
    done = set(pd.read_csv(RESULTS_CSV)["run_id"]) if os.path.exists(RESULTS_CSV) else set()
    start = time.time()
    durations = []
    for name, seed in RUN_PLAN:
        run_id = f"{name}_s{seed}"
        if run_id in done:
            print(f"skip {run_id} (already in results)")
            continue
        elapsed_h = (time.time() - start) / 3600
        est_h = (max(durations) if durations else 0) / 60
        if elapsed_h + est_h > TIME_BUDGET_HOURS:
            print(f"Stopping: {elapsed_h:.1f} h used, next run ~{est_h:.1f} h would exceed "
                  f"the {TIME_BUDGET_HOURS} h budget. Save {OUT_DIR} and re-run in a new session.")
            break
        print(f"\n=== {run_id} ===  config: {EXPERIMENTS[name]}")
        row = train_one(name, seed)
        append_result(row)
        durations.append(row["train_minutes"])
    print("\nDone for this session. Run MODE = 'report' for the comparison.")


# ----------------------------------------------------------------------
# Report
# ----------------------------------------------------------------------
def report(out_dir=OUT_DIR, ann_dir=ANN_DIR, make_chart=True, n_boot=None):
    n_boot = n_boot or N_BOOTSTRAP
    res = pd.read_csv(os.path.join(out_dir, "results.csv"))
    test_ann = f"{ann_dir}/instances_test.json"
    with open(test_ann) as f:
        img_ids = [im["id"] for im in json.load(f)["images"]]
    lines = []
    P = lambda s="": (print(s), lines.append(s))
    recipes = [r for r in EXPERIMENTS if r in set(res.recipe)]
    suffixes = ("", "_tta")

    # Per-image matches at every IoU threshold, for every run (plain + TTA)
    matches = {}
    for run_id in res.run_id:
        for sfx in suffixes:
            p = os.path.join(out_dir, f"preds_{run_id}{sfx}.parquet")
            if os.path.exists(p):
                pr = pd.read_parquet(p)
                for t in IOU_THRS:
                    matches[(run_id + sfx, t)] = per_image_matches(test_ann, pr, iou_thr=t)

    def run_ap(key, ids, metric):
        thrs = [0.5] if metric == "ap50" else IOU_THRS
        return np.mean([ap_from_matches(matches[(key, t)], ids) for t in thrs])

    def recipe_ap(recipe, sfx, ids, metric):
        keys = [r + sfx for r in res[res.recipe == recipe].run_id if (r + sfx, 0.5) in matches]
        return np.mean([run_ap(k, ids, metric) for k in keys])

    best = res.sort_values("test_ap50").iloc[-1]
    c50, c5095 = run_ap(best.run_id, img_ids, "ap50"), run_ap(best.run_id, img_ids, "ap5095")

    rng = np.random.default_rng(0)
    samples = [rng.choice(img_ids, size=len(img_ids), replace=True) for _ in range(n_boot)]
    boot = {(r, sfx, m): np.array([recipe_ap(r, sfx, s_, m) for s_ in samples])
            for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}
    ci = {k: np.nanpercentile(v, [2.5, 97.5]) for k, v in boot.items()}
    col = {("", "ap50"): "test_ap50", ("_tta", "ap50"): "test_ap50_tta",
           ("", "ap5095"): "test_ap50_95", ("_tta", "ap5095"): "test_ap50_95_tta"}
    mean = {(r, sfx, m): res.loc[res.recipe == r, col[(sfx, m)]].mean()
            for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}
    sd = {(r, sfx, m): res.loc[res.recipe == r, col[(sfx, m)]].std()
          for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}

    P("=" * 78)
    P("INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)")
    P("=" * 78)
    P("mean over seeds ± sd across seeds   [95% bootstrap range over test images]")
    P()
    P(f"{'':34s} {'mAP@0.5':>26s}   {'mAP@0.5:0.95':>26s}")
    for r in recipes:
        for sfx in suffixes:
            name = RECIPE_LABELS.get(r, r) + (" + TTA" if sfx else "")
            cells = []
            for m in ("ap50", "ap5095"):
                lo, hi = ci[(r, sfx, m)]
                s_ = sd[(r, sfx, m)]
                cells.append(f"{mean[(r, sfx, m)]:5.2f} ± {s_:4.2f} [{lo:5.1f}-{hi:5.1f}]"
                             if pd.notna(s_) else f"{mean[(r, sfx, m)]:5.2f}        [{lo:5.1f}-{hi:5.1f}]")
            P(f"{name:34s} {cells[0]:>26s}   {cells[1]:>26s}")
    P()
    P(f"Metric check on {best.run_id}: fast mAP@0.5 {c50:.2f} vs pycocotools {best.test_ap50:.2f}; "
      f"fast mAP@0.5:0.95 {c5095:.2f} vs pycocotools {best.test_ap50_95:.2f} (should agree within ~0.5)")
    P()
    P("Individual runs:")
    P(res[["run_id", "best_epoch", "val_ap50", "test_ap50", "test_ap50_95",
           "test_ap50_tta"]].round(2).to_string(index=False))

    if "baseline" in recipes:
        P()
        P("Better than our original training? (paired bootstrap of the difference;")
        P("'yes' only if the whole 95% range is above 0)")
        for m, mname in (("ap50", "mAP@0.5"), ("ap5095", "mAP@0.5:0.95")):
            P(f"  {mname}:")
            for r in recipes:
                for sfx in suffixes:
                    if r == "baseline" and sfx == "":
                        continue
                    d = boot[(r, sfx, m)] - boot[("baseline", "", m)]
                    lo, hi = np.nanpercentile(d, [2.5, 97.5])
                    v = "yes" if lo > 0 else ("no -- worse" if hi < 0 else "not distinguishable")
                    P(f"    {RECIPE_LABELS.get(r, r) + (' + TTA' if sfx else ''):40s} {np.nanmean(d):+6.2f} "
                      f"(95%: {lo:+.2f} to {hi:+.2f}) -> {v}")

    P()
    P("Against the published numbers (no TTA, mean over our seeds). 'higher' / 'lower'")
    P("only when the published value lies outside our 95% range; otherwise 'tie'.")
    P("Valid only if the paper used the same 74-image S1-IW-2023 test split.")
    for pname, p50, p5095 in PUBLISHED:
        P(f"  {pname} (mAP@0.5 {p50:.2f}" + (f", mAP@0.5:0.95 {p5095:.2f})" if p5095 else ")"))
        for r in recipes:
            out = []
            for m, pv in (("ap50", p50), ("ap5095", p5095)):
                if pv is None:
                    out.append("n/r")
                    continue
                lo, hi = ci[(r, "", m)]
                out.append("higher" if lo > pv else ("lower" if hi < pv else "tie"))
            P(f"    {RECIPE_LABELS.get(r, r):40s} mAP@0.5: {out[0]:6s}  mAP@0.5:0.95: {out[1]}")

    with open(os.path.join(out_dir, "report.txt"), "w") as f:
        f.write("\n".join(lines))
    if make_chart:
        _charts(out_dir, recipes, mean, ci)
        P(f"\nSaved report.txt, comparison_chart.png (detailed) and presentation_chart.png in {out_dir}")


def _charts(out_dir, recipes, mean, ci):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    OURS, OTHER, INK, INK2, SURF = "#2a78d6", "#b9b8b2", "#0b0b0b", "#52514e", "#fcfcfb"
    plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2,
                         "xtick.color": INK2, "ytick.color": INK})

    # 1) Detailed: every recipe, with and without TTA, mAP@0.5 only
    rows = []
    for r in recipes:
        for sfx in ("", "_tta"):
            rows.append((RECIPE_LABELS.get(r, r) + (" + TTA" if sfx else ""), mean[(r, sfx, "ap50")],
                         ci[(r, sfx, "ap50")], OURS if r != "baseline" else "#9fb9d9"))
    rows += [(n, v, None, OTHER) for n, v, _ in PUBLISHED]
    fig, ax = plt.subplots(figsize=(9.5, 0.45 * len(rows) + 1.4), facecolor=SURF)
    _bars(ax, rows, "mAP@0.5 (%)", INK)
    ax.set_title("Internal-wave detection on the S1-IW-2023 test set (all variants)", loc="left", fontsize=11)
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "comparison_chart.png"), dpi=130, facecolor=SURF)
    plt.close(fig)

    # 2) Presentation: one bar per model, both metrics, sorted by mAP@0.5
    rows = [(RECIPE_LABELS.get(r, r), r) for r in recipes]
    entries = ([(lbl, mean[(r, "", "ap50")], ci[(r, "", "ap50")], mean[(r, "", "ap5095")],
                 ci[(r, "", "ap5095")], OURS) for lbl, r in rows]
               + [(n, a, None, b, None, OTHER) for n, a, b in PUBLISHED])
    entries.sort(key=lambda e: e[1], reverse=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.5 * len(entries) + 2.0), facecolor=SURF, sharey=True)
    _bars(axes[0], [(e[0], e[1], e[2], e[5]) for e in entries], "mAP@0.5 (%)", INK)
    _bars(axes[1], [(e[0], e[3], e[4], e[5]) for e in entries], "mAP@0.5:0.95 (%)  -- stricter: rewards tight boxes",
          INK, show_labels=False)
    axes[0].set_title("Detection (box overlaps the wave by at least 50%)", loc="left", fontsize=11)
    axes[1].set_title("Detection + precise box placement", loc="left", fontsize=11)
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(color=OURS, label="Our models (mean of 3 runs; line = 95% range)"),
                        Patch(color=OTHER, label="Published results, same dataset")],
               loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle("Internal-wave detection on the S1-IW-2023 test set", x=0.01, ha="left",
                 fontsize=13, fontweight="bold")
    fig.subplots_adjust(left=0.2, right=0.98, top=0.86, bottom=0.17, wspace=0.08)
    fig.savefig(os.path.join(out_dir, "presentation_chart.png"), dpi=150, facecolor=SURF)
    plt.close(fig)


def _bars(ax, rows, xlabel, ink, show_labels=True):
    """rows: (label, value or None, (lo, hi) or None, color). Top row first."""
    y = np.arange(len(rows))[::-1]
    vals = [r[1] for r in rows if r[1] is not None]
    lo_x = max(0, np.floor((min(vals) - 8) / 5) * 5)
    for yy, (lbl, v, rng_, c) in zip(y, rows):
        if v is None:
            ax.text(lo_x + 0.5, yy, "not reported", va="center", fontsize=9, color="#8a8984", style="italic")
            continue
        ax.barh(yy, v - lo_x, left=lo_x, height=0.62, color=c, edgecolor="none")
        if rng_ is not None:
            ax.plot(rng_, [yy, yy], color=ink, lw=1.2)
            ax.plot([rng_[0]] * 2, [yy - 0.13, yy + 0.13], color=ink, lw=1.2)
            ax.plot([rng_[1]] * 2, [yy - 0.13, yy + 0.13], color=ink, lw=1.2)
        x_txt = (rng_[1] if rng_ is not None else v) + 0.4
        ax.text(x_txt, yy, f"{v:.1f}", va="center", fontsize=9, color=ink)
    if show_labels:
        ax.set_yticks(y)
        ax.set_yticklabels([r[0] for r in rows])
    else:
        ax.tick_params(labelleft=False)
    ax.set_xlim(lo_x, 100)
    ax.set_ylim(-0.6, len(rows) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(axis="y", length=0)


# ----------------------------------------------------------------------
# Self-test (CPU, synthetic, ~1-2 min)
# ----------------------------------------------------------------------
def selftest():
    import tempfile
    print("=" * 70 + "\nSELF-TEST\n" + "=" * 70)
    ok_all = True

    print("\n[1] Augmentation keeps boxes on the objects (all 8 orientations)")
    torch.manual_seed(0); random.seed(0)
    worst = 0.0
    for _ in range(64):
        img = torch.zeros(3, 120, 200)
        bx = torch.tensor([[30., 20., 90., 50.], [120., 60., 180., 110.]])
        for x1, y1, x2, y2 in bx.int().tolist():
            img[:, y1:y2, x1:x2] = 1.0
        a_img, a_b = augment(img, bx)
        m = a_img[0] > 0.5
        for x1, y1, x2, y2 in a_b.round().int().tolist():
            inside = m[y1:y2, x1:x2].float().mean().item()
            worst = max(worst, 1 - inside)
        if abs(m.sum().item() - sum(((b[2] - b[0]) * (b[3] - b[1])).item() for b in a_b)) > 1:
            worst = 1.0
    ok = worst < 0.01
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  worst box/object mismatch {worst:.3f}")

    print("\n[2] Fast mAP (used for the confidence ranges) matches pycocotools")
    tmp = tempfile.mkdtemp()
    rng = np.random.default_rng(1)
    images, anns, preds = [], [], []
    aid = 1
    for iid in range(1, 41):
        images.append(dict(id=iid, file_name=f"{iid}.png", width=256, height=256))
        for _ in range(rng.integers(1, 4)):
            x, y = rng.uniform(0, 180, 2); w, h = rng.uniform(20, 70, 2)
            anns.append(dict(id=aid, image_id=iid, category_id=1, bbox=[x, y, w, h], area=w * h, iscrowd=0))
            aid += 1
            if rng.random() < 0.8:
                j = rng.normal(0, 6, 4)
                preds.append(dict(image_id=iid, x1=x + j[0], y1=y + j[1], x2=x + w + j[2],
                                  y2=y + h + j[3], score=rng.uniform(0.3, 1)))
        for _ in range(rng.integers(0, 3)):
            x, y = rng.uniform(0, 200, 2)
            preds.append(dict(image_id=iid, x1=x, y1=y, x2=x + 40, y2=y + 40, score=rng.uniform(0, 0.8)))
    ann_path = os.path.join(tmp, "t.json")
    with open(ann_path, "w") as f:
        json.dump(dict(images=images, annotations=anns, categories=[dict(id=1, name="w")]), f)
    preds = pd.DataFrame(preds)
    ref, ref95 = coco_eval(ann_path, preds)
    ids = [im["id"] for im in images]
    fast = ap_from_matches(per_image_matches(ann_path, preds), ids)
    fast95 = np.mean([ap_from_matches(per_image_matches(ann_path, preds, iou_thr=t), ids) for t in IOU_THRS])
    ok = abs(ref - fast) < 0.5 and abs(ref95 - fast95) < 0.5
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  mAP@0.5: pycocotools {ref:.2f} vs fast {fast:.2f} | "
          f"mAP@0.5:0.95: pycocotools {ref95:.2f} vs fast {fast95:.2f}")

    print("\n[3] Test-time augmentation maps flipped boxes back correctly")
    class Fake(torch.nn.Module):
        def forward(self, xs):                      # 'detects' the bright square
            outs = []
            for x in xs:
                ys, xs_ = torch.where(x[0] > 0.5)
                b = torch.tensor([[xs_.min(), ys.min(), xs_.max() + 1, ys.max() + 1]], dtype=torch.float32)
                outs.append(dict(boxes=b, scores=torch.tensor([0.9])))
            return outs
    img = torch.zeros(3, 100, 160); img[:, 10:30, 20:70] = 1
    b, s = predict(Fake().eval(), img, tta=True)
    ok = len(b) == 1 and torch.allclose(b[0], torch.tensor([20., 10., 70., 30.]), atol=0.5) and abs(s.item() - 0.9) < 1e-4
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  fused box {b.tolist()} score {s.tolist()}")

    print("\n[4] Faster R-CNN v2 builds and runs")
    try:
        m2 = build_model("v2", 192, pretrained=False).eval()
        with torch.no_grad():
            o = m2([torch.rand(3, 192, 192)])[0]
        ok = set(o) >= {"boxes", "scores", "labels"} and m2.roi_heads.box_predictor.cls_score.out_features == 2
    except Exception as e:
        print("  ", e); ok = False
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  v2 model forward pass, 2 output classes")

    print("\n[5] End-to-end: tiny synthetic dataset, 2 recipes x 2 seeds, train -> report (CPU)")
    img_dir = os.path.join(tmp, "img"); os.makedirs(img_dir)
    ann_dir = os.path.join(tmp, "ann"); os.makedirs(ann_dir)
    rng = np.random.default_rng(2)
    for split, n in (("train", 12), ("val", 4), ("test", 6)):
        ims, ans = [], []
        for k in range(n):
            iid = {"train": 0, "val": 100, "test": 200}[split] + k
            arr = (rng.normal(0.35, 0.08, (192, 192)) * 255).clip(0, 255).astype(np.uint8)
            x, y = rng.integers(10, 110, 2); w, h = rng.integers(40, 70, 2)
            for c in range(0, w, 12):                   # bright stripes = a 'wave packet'
                arr[y:y + h, x + c:x + c + 5] = 230
            Image.fromarray(arr).save(os.path.join(img_dir, f"{iid}.png"))
            ims.append(dict(id=iid, file_name=f"{iid}.png", width=192, height=192))
            ans.append(dict(id=iid + 1, image_id=iid, category_id=1, bbox=[int(x), int(y), int(w), int(h)],
                            area=int(w * h), iscrowd=0))
        with open(os.path.join(ann_dir, f"instances_{split}.json"), "w") as f:
            json.dump(dict(images=ims, annotations=ans, categories=[dict(id=1, name="internal wave")]), f)
    out = os.path.join(tmp, "out"); os.makedirs(out)
    global NUM_WORKERS, BATCH_SIZE
    nw, bs = NUM_WORKERS, BATCH_SIZE
    NUM_WORKERS, BATCH_SIZE = 0, 2
    try:
        for name, seed in [("baseline", 0), ("improved", 0), ("baseline", 1), ("improved", 1)]:
            row = train_one(name, seed, ann_dir=ann_dir, image_dir=img_dir, out_dir=out, pretrained=False,
                            cfg_override=dict(epochs=2, img_size=192), log_every=1000)
            append_result(row, os.path.join(out, "results.csv"))
        report(out_dir=out, ann_dir=ann_dir, make_chart=True, n_boot=200)
        ok = all(os.path.exists(os.path.join(out, f)) for f in ("comparison_chart.png", "presentation_chart.png"))
    except Exception as e:
        import traceback; traceback.print_exc()
        ok = False
    finally:
        NUM_WORKERS, BATCH_SIZE = nw, bs
    ok_all &= ok
    print(f"\n  {'PASS' if ok else 'FAIL'}  pipeline ran end to end (numbers above are meaningless --"
          f" untrained model, 2 epochs, synthetic data)")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- fix before training.")
    print("=" * 70)
    return ok_all


def main():
    in_nb = "ipykernel" in sys.modules
    mode = MODE
    if not in_nb and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "train", "report"):
        mode = sys.argv[1]
    print(f"Mode: {mode} | device: {DEVICE}")
    if mode == "selftest":
        selftest()
    elif mode == "train":
        train_all()
    elif mode == "report":
        report()
    else:
        print("MODE must be 'selftest', 'train' or 'report'")


if __name__ == "__main__":
    main()

Mode: report | device: cuda
INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)
mean over seeds ± sd across seeds   [95% bootstrap range over test images]

                                                      mAP@0.5                 mAP@0.5:0.95
Ours: original training            85.02 ± 1.43 [ 79.8- 90.7]   57.75 ± 1.69 [ 52.7- 63.5]
Ours: original training + TTA      88.63 ± 1.42 [ 83.9- 93.4]   62.08 ± 1.11 [ 57.8- 67.1]
Ours: improved training            91.08 ± 0.66 [ 86.9- 95.0]   61.26 ± 1.85 [ 56.8- 66.2]
Ours: improved training + TTA      90.38 ± 0.35 [ 86.3- 94.4]   62.92 ± 0.86 [ 58.3- 68.1]
Ours: improved training, v2 model  92.27 ± 1.13 [ 88.3- 96.1]   64.38 ± 2.14 [ 60.1- 69.1]
Ours: improved training, v2 model + TTA 91.94 ± 1.35 [ 88.1- 95.6]   67.73 ± 0.49 [ 63.3- 72.7]

Metric check on improved_v2_s0: fast mAP@0.5 93.57 vs pycocotools 93.57; fast mAP@0.5:0.95 66.74 vs pycocotools 66.74 (should agree within ~0.5)

Individual runs:
        run_id  best_ep

In [3]:
"""
OceanXRay -- Improving the internal-wave detector, measured honestly
=====================================================================
Trains several detector "recipes" on S1-IW-2023 (same 593/75/74 split as
before), each with 3 random seeds, and reports mAP on the held-out test
set as mean +/- spread, plus bootstrap confidence intervals. With only 74
test images (108 waves), a 1-2 point difference can be pure luck; this
script tells you which differences are real.

RECIPES (edit EXPERIMENTS below to add more):
  baseline     -- the old recipe's settings: Faster R-CNN v1, no
                  augmentation, images shrunk to 800 px (torchvision
                  default), fixed LR, last epoch kept. Re-run here so the
                  comparison is fair (same code, same GPU, same seeds).
  improved     -- same model, plus:
                    * flips + 90-degree rotations (waves can travel any way,
                      so every training image counts 8x) + light brightness
                      / contrast jitter
                    * full 1024 px resolution (no shrinking)
                    * warm-up + cosine learning-rate schedule
                    * keep the epoch that scores best on the VALIDATION set
  improved_v2  -- same as 'improved' with torchvision's Faster R-CNN v2
                  (newer, stronger training recipe of the same design)
  retinanet / fcos / ssd -- competitor designs (as in the reference paper's
                  comparison table), trained by us on the SAME split with
                  the SAME improved recipe: the fair comparison. The paper's
                  own numbers come from a different evaluation set (it added
                  1,133 unreleased images and re-split), so they are listed
                  for context only.

Every run is also evaluated with test-time augmentation (TTA): predict on
4 flipped copies of each test image and merge -- reported as a separate
column, never mixed with the plain numbers.

HOW TO RUN (Kaggle, GPU on)
---------------------------
  MODE = "selftest"  -- 1-2 min on CPU, tiny synthetic data; checks the
                        augmentation, metric and full pipeline. Run first.
  MODE = "train"     -- works through RUN_PLAN, skipping runs already in
                        RESULTS_CSV. Stops starting new runs before
                        TIME_BUDGET_HOURS, so it is safe to re-launch in a
                        new Kaggle session: it continues where it stopped
                        (copy OUT_DIR between sessions, e.g. as a dataset).
  MODE = "report"    -- comparison table, bootstrap intervals, bar chart.

Outputs (OUT_DIR): results.csv, preds_<run>.parquet (test predictions),
ckpt_<run>.pth (best weights), report.txt, comparison_chart.png
"""

import os
import sys
import json
import math
import time
import random

import numpy as np
import pandas as pd
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

# =========================== EDIT THESE ===========================
MODE = "report"            # "selftest" -> "train" -> "report"

DATASET_ROOT = "/kaggle/input/datasets/mayankgupta005/s1-iw-2023-datasetocean-internal-wave-dataset"
ANN_DIR = f"{DATASET_ROOT}/Annotation/Annotation"
IMAGE_DIR = f"{DATASET_ROOT}/IW_Image/IW_Image"
OUT_DIR = "outputs/iw_improve"

TIME_BUDGET_HOURS = 11.0     # Kaggle sessions end at 12 h
BATCH_SIZE = 4
NUM_WORKERS = 2

# Published results to show on the chart, for the SAME dataset/split only.
# Fill these from the papers; leave out anything not measured on S1-IW-2023.
# (name, mAP@0.5, mAP@0.5:0.95 or None if not reported) -- from the reference
# paper's Tables 2-4. CONTEXT ONLY, NOT COMPARABLE: the paper added 1,133 of
# its own (unreleased) images to S1-IW-2023 and re-split everything 8:2, so
# its scores come from a different, larger evaluation set (and it trained on
# ~2.5x more images). The paper also gives its own model 91.20 (Tables 3, 4)
# and 90.57 (Table 2) with otherwise identical numbers.
PUBLISHED_NOTE = ("Paper's numbers are from a different evaluation set (S1-IW-2023 + 1,133 "
                  "unreleased images, random 8:2 re-split) -- shown for context, not compared.")
PUBLISHED = [
    ("Paper: rFaster R-CNN (their model)", 91.20, 58.80),
    ("Paper: Faster R-CNN (R50 + FPN)",    87.17, 55.10),
    ("Paper: YOLOv9",                       87.80, None),
    ("Paper: RetinaNet + FPN",              84.86, None),
    ("Paper: SSD",                          84.64, None),
    ("Paper: YOLOv3",                       71.50, None),
]
RECIPE_LABELS = {"baseline": "Faster R-CNN, original training",
                 "improved": "Faster R-CNN, improved training (ours)",
                 "improved_v2": "Faster R-CNN v2, improved training (ours)",
                 "retinanet": "RetinaNet (R50-FPN v2), same training",
                 "fcos": "FCOS (R50-FPN), same training",
                 "ssd": "SSD300 (VGG16), same training"}
OUR_RECIPES = ("improved", "improved_v2")   # highlighted; compared against all others
IOU_THRS = np.round(np.linspace(0.5, 0.95, 10), 2)
# ==================================================================

EXPERIMENTS = {
    # baseline = the settings of train_iw_detector.py (SGD lr 0.005, momentum
    # 0.9, weight decay 5e-4, no warm-up, no clipping, last epoch kept).
    "baseline":    dict(arch="v1", augment=False, img_size=800,  epochs=20, lr=0.005,
                        schedule="constant", weight_decay=5e-4, clip=None, select="last"),
    "improved":    dict(arch="v1", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    "improved_v2": dict(arch="v2", augment=True,  img_size=1024, epochs=24, lr=0.005,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    # Competitors from the paper's comparison table, trained by us on the SAME
    # public split with the SAME improved recipe -- a genuinely fair comparison.
    # LRs follow torchvision's reference settings scaled to batch size 4.
    # (YOLO is not in torchvision; it needs the separate 'ultralytics' package.)
    "retinanet":   dict(arch="retinanet", augment=True, img_size=1024, epochs=24, lr=0.0025,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    "fcos":        dict(arch="fcos", augment=True, img_size=1024, epochs=24, lr=0.0025,
                        schedule="cosine", weight_decay=1e-4, clip=10.0, select="best_val"),
    # SSD300 resizes every image to 300 x 300 by design (img_size is ignored).
    "ssd":         dict(arch="ssd", augment=True, img_size=300, epochs=24, lr=0.002,
                        schedule="cosine", weight_decay=5e-4, clip=10.0, select="best_val"),
}
# Order matters: the most informative comparison first, so a single
# Kaggle session already answers "does the new recipe beat the old one?"
RUN_PLAN = [("baseline", 0), ("improved", 0),
            ("baseline", 1), ("improved", 1),
            ("baseline", 2), ("improved", 2),
            ("improved_v2", 0), ("improved_v2", 1), ("improved_v2", 2),
            ("retinanet", 0), ("fcos", 0), ("ssd", 0),
            ("retinanet", 1), ("fcos", 1), ("ssd", 1),
            ("retinanet", 2), ("fcos", 2), ("ssd", 2)]

VAL_EVERY = 2                # epochs between validation checks
N_BOOTSTRAP = 2000
NUM_CLASSES = 2              # background + internal wave
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_CSV = os.path.join(OUT_DIR, "results.csv")


# ----------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------
class COCODetectionDataset(Dataset):
    """Same loader as train_iw_detector.py, plus optional augmentation."""

    def __init__(self, annotations_path, image_dir, augment=False):
        with open(annotations_path) as f:
            coco = json.load(f)
        self.image_dir = image_dir
        self.augment = augment
        self.images = {img["id"]: img for img in coco["images"]}
        self.anns = {}
        for a in coco["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        self.image_ids = list(self.images.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        info = self.images[image_id]
        img = TF.to_tensor(Image.open(os.path.join(self.image_dir, info["file_name"])).convert("RGB"))
        boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]]
                 for a in self.anns.get(image_id, [])]
        labels = [a["category_id"] for a in self.anns.get(image_id, [])]
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        if self.augment:
            img, boxes = augment(img, boxes)
        return img, {"boxes": boxes,
                     "labels": torch.tensor(labels, dtype=torch.int64),
                     "image_id": torch.tensor([image_id])}


def augment(img, boxes):
    """All 8 flips/90-degree rotations (via transpose + flips, so box maths
    is exact) plus mild brightness/contrast jitter. boxes: [N,4] xyxy."""
    _, H, W = img.shape
    b = boxes.clone()
    if random.random() < 0.5:                      # transpose (swap x and y)
        img = img.transpose(1, 2)
        b = b[:, [1, 0, 3, 2]]
        H, W = W, H
    if random.random() < 0.5:                      # horizontal flip
        img = img.flip(-1)
        b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], dim=1)
    if random.random() < 0.5:                      # vertical flip
        img = img.flip(-2)
        b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], dim=1)
    if random.random() < 0.8:                      # brightness / contrast
        c = random.uniform(0.85, 1.15)
        br = random.uniform(-0.08, 0.08)
        m = img.mean()
        img = ((img - m) * c + m + br).clamp(0, 1)
    return img.contiguous(), b


def collate_fn(batch):
    return tuple(zip(*batch))


# ----------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------
def build_model(arch, img_size, pretrained=True):
    """COCO-pretrained detector with its class-prediction head replaced for
    our 2 classes (background + internal wave)."""
    D = torchvision.models.detection
    fns = {"v1": D.fasterrcnn_resnet50_fpn, "v2": D.fasterrcnn_resnet50_fpn_v2,
           "retinanet": D.retinanet_resnet50_fpn_v2, "fcos": D.fcos_resnet50_fpn,
           "ssd": D.ssd300_vgg16}
    if arch not in fns:
        raise ValueError(arch)
    kw = {} if arch == "ssd" else dict(min_size=img_size, max_size=img_size)
    w = dict(weights="DEFAULT") if pretrained else dict(weights=None, weights_backbone=None)
    model = fns[arch](**w, **kw)
    if arch in ("v1", "v2"):
        in_f = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_f, NUM_CLASSES)
    elif arch == "retinanet":
        from torchvision.models.detection.retinanet import RetinaNetClassificationHead
        head = model.head.classification_head
        model.head.classification_head = RetinaNetClassificationHead(
            256, head.num_anchors, NUM_CLASSES, norm_layer=lambda c: torch.nn.GroupNorm(32, c))
    elif arch == "fcos":
        from torchvision.models.detection.fcos import FCOSClassificationHead
        model.head.classification_head = FCOSClassificationHead(
            256, model.head.classification_head.num_anchors, NUM_CLASSES)
    elif arch == "ssd":
        from torchvision.models.detection.ssd import SSDClassificationHead
        from torchvision.models.detection import _utils as det_utils
        ch = det_utils.retrieve_out_channels(model.backbone, (300, 300))
        model.head.classification_head = SSDClassificationHead(
            ch, model.anchor_generator.num_anchors_per_location(), NUM_CLASSES)
    return model


def lr_at(epoch_float, cfg):
    if cfg["schedule"] == "constant":          # old recipe: fixed LR, no warm-up
        return cfg["lr"]
    warm = 1.0
    if epoch_float < warm:
        return cfg["lr"] * (0.1 + 0.9 * epoch_float / warm)
    t = (epoch_float - warm) / max(cfg["epochs"] - warm, 1e-6)
    return cfg["lr"] * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(t, 1.0))))


# ----------------------------------------------------------------------
# Inference (+ test-time augmentation)
# ----------------------------------------------------------------------
@torch.no_grad()
def predict(model, img, tta=False):
    """Returns (boxes [N,4] xyxy, scores [N]) on CPU."""
    _, H, W = img.shape
    outs = []
    variants = [(False, False)] + ([(True, False), (False, True), (True, True)] if tta else [])
    for hf, vf in variants:
        x = img
        if hf:
            x = x.flip(-1)
        if vf:
            x = x.flip(-2)
        o = model([x.to(DEVICE)])[0]
        b = o["boxes"].float().cpu()
        if hf:
            b = torch.stack([W - b[:, 2], b[:, 1], W - b[:, 0], b[:, 3]], 1)
        if vf:
            b = torch.stack([b[:, 0], H - b[:, 3], b[:, 2], H - b[:, 1]], 1)
        outs.append((b, o["scores"].float().cpu()))
    boxes = torch.cat([o[0] for o in outs])
    scores = torch.cat([o[1] for o in outs])
    if tta and len(boxes):
        boxes, scores = fuse_boxes(boxes, scores, n_views=len(variants))
    return boxes, scores


def fuse_boxes(boxes, scores, n_views, iou_thr=0.55):
    """Weighted box fusion (simple version): cluster overlapping boxes from
    the flipped views, average their coordinates weighted by score, and
    scale the score by how many views agreed."""
    order = scores.argsort(descending=True)
    boxes, scores = boxes[order], scores[order]
    used = torch.zeros(len(boxes), dtype=torch.bool)
    fused_b, fused_s = [], []
    iou = torchvision.ops.box_iou(boxes, boxes)
    for i in range(len(boxes)):
        if used[i]:
            continue
        grp = (~used) & (iou[i] >= iou_thr)
        grp[i] = True
        used |= grp
        w = scores[grp]
        fused_b.append((boxes[grp] * w[:, None]).sum(0) / w.sum())
        fused_s.append(w.mean() * min(int(grp.sum()), n_views) / n_views)
    return torch.stack(fused_b), torch.stack(fused_s)


def predict_split(model, ds, tta=False):
    model.eval()
    rows = []
    for idx in range(len(ds)):
        img, tgt = ds[idx]
        b, s = predict(model, img, tta=tta)
        iid = int(tgt["image_id"])
        for bb, ss in zip(b.tolist(), s.tolist()):
            rows.append(dict(image_id=iid, x1=bb[0], y1=bb[1], x2=bb[2], y2=bb[3], score=ss))
    return pd.DataFrame(rows, columns=["image_id", "x1", "y1", "x2", "y2", "score"])


# ----------------------------------------------------------------------
# Metrics
# ----------------------------------------------------------------------
def coco_eval(ann_path, preds):
    """Official pycocotools numbers: returns (AP50, AP50:95) in %."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    if len(preds) == 0:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(ann_path)
        res = [dict(image_id=int(r.image_id), category_id=1,
                    bbox=[r.x1, r.y1, r.x2 - r.x1, r.y2 - r.y1], score=r.score)
               for r in preds.itertuples()]
        dt = gt.loadRes(res)
        ev = COCOeval(gt, dt, iouType="bbox")
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return 100 * ev.stats[1], 100 * ev.stats[0]


def per_image_matches(ann_path, preds, iou_thr=0.5, max_dets=100):
    """Greedy COCO-style matching at one IoU. Returns {image_id: (scores,
    is_tp, n_gt)} so AP can be recomputed quickly on bootstrap samples."""
    with open(ann_path) as f:
        coco = json.load(f)
    gt = {}
    for a in coco["annotations"]:
        x, y, w, h = a["bbox"]
        gt.setdefault(a["image_id"], []).append([x, y, x + w, y + h])
    out = {}
    for img in coco["images"]:
        iid = img["id"]
        g = torch.tensor(gt.get(iid, []), dtype=torch.float32).reshape(-1, 4)
        p = preds[preds.image_id == iid].sort_values("score", ascending=False).head(max_dets)
        sc = p.score.to_numpy()
        tp = np.zeros(len(p), dtype=bool)
        if len(p) and len(g):
            ious = torchvision.ops.box_iou(torch.tensor(p[["x1", "y1", "x2", "y2"]].to_numpy(),
                                                        dtype=torch.float32), g).numpy()
            taken = np.zeros(len(g), dtype=bool)
            for k in range(len(p)):
                cand = np.where(~taken & (ious[k] >= iou_thr))[0]
                if len(cand):
                    j = cand[np.argmax(ious[k, cand])]
                    taken[j] = True
                    tp[k] = True
        out[iid] = (sc, tp, len(g))
    return out


def ap_from_matches(matches, image_ids):
    """COCO 101-point interpolated AP for a (possibly repeated) list of images."""
    sc = np.concatenate([matches[i][0] for i in image_ids]) if len(image_ids) else np.array([])
    tp = np.concatenate([matches[i][1] for i in image_ids]) if len(image_ids) else np.array([])
    n_gt = sum(matches[i][2] for i in image_ids)
    if n_gt == 0:
        return np.nan
    order = np.argsort(-sc, kind="mergesort")
    tp = tp[order].astype(float)
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    for k in range(len(prec) - 2, -1, -1):          # make precision monotone
        prec[k] = max(prec[k], prec[k + 1])
    rs = np.linspace(0, 1, 101)
    idx = np.searchsorted(rec, rs, side="left")
    return 100 * np.mean([prec[i] if i < len(prec) else 0.0 for i in idx])


# ----------------------------------------------------------------------
# Training one run
# ----------------------------------------------------------------------
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one(name, seed, ann_dir=ANN_DIR, image_dir=IMAGE_DIR, out_dir=OUT_DIR,
              pretrained=True, cfg_override=None, log_every=20):
    cfg = dict(EXPERIMENTS[name], **(cfg_override or {}))
    run_id = f"{name}_s{seed}"
    set_seed(seed)
    tr = COCODetectionDataset(f"{ann_dir}/instances_train.json", image_dir, augment=cfg["augment"])
    va = COCODetectionDataset(f"{ann_dir}/instances_val.json", image_dir)
    te = COCODetectionDataset(f"{ann_dir}/instances_test.json", image_dir)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
                    num_workers=NUM_WORKERS, generator=g, drop_last=True)
    model = build_model(cfg["arch"], cfg["img_size"], pretrained).to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=cfg["lr"], momentum=0.9, weight_decay=cfg["weight_decay"])
    use_amp = DEVICE.type == "cuda"     # mixed precision on GPU for speed (all recipes alike)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ckpt_path = os.path.join(out_dir, f"ckpt_{run_id}.pth")
    best_val, best_epoch, history = -1.0, -1, []
    t0 = time.time()
    steps_per_epoch = len(dl)
    for epoch in range(cfg["epochs"]):
        model.train()
        ep_loss = 0.0
        for step, (imgs, tgts) in enumerate(dl):
            for pg in opt.param_groups:
                pg["lr"] = lr_at(epoch + step / steps_per_epoch, cfg)
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                loss = sum(model(imgs, tgts).values())
            opt.zero_grad()
            scaler.scale(loss).backward()
            if cfg["clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update()
            ep_loss += loss.item()
            if step % log_every == 0:
                print(f"  [{run_id}] epoch {epoch + 1}/{cfg['epochs']} step {step}/{steps_per_epoch} "
                      f"loss {loss.item():.4f} lr {opt.param_groups[0]['lr']:.5f}")
        rec = dict(epoch=epoch + 1, loss=ep_loss / max(steps_per_epoch, 1),
                   minutes=(time.time() - t0) / 60)
        last = epoch == cfg["epochs"] - 1
        if cfg["select"] == "best_val" and ((epoch + 1) % VAL_EVERY == 0 or last):
            v50, _ = coco_eval(f"{ann_dir}/instances_val.json", predict_split(model, va))
            rec["val_ap50"] = v50
            if v50 > best_val:
                best_val, best_epoch = v50, epoch + 1
                torch.save(model.state_dict(), ckpt_path)
        if cfg["select"] == "last" and last:
            torch.save(model.state_dict(), ckpt_path)
            best_epoch = epoch + 1
        history.append(rec)
        print(f"  [{run_id}] epoch {epoch + 1} done: loss {rec['loss']:.4f}"
              + (f", val AP50 {rec['val_ap50']:.2f}" if "val_ap50" in rec else "")
              + f"  ({rec['minutes']:.1f} min so far)")
    pd.DataFrame(history).to_csv(os.path.join(out_dir, f"history_{run_id}.csv"), index=False)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_ann = f"{ann_dir}/instances_test.json"
    p_plain = predict_split(model, te, tta=False)
    p_tta = predict_split(model, te, tta=True)
    p_plain.to_parquet(os.path.join(out_dir, f"preds_{run_id}.parquet"), index=False)
    p_tta.to_parquet(os.path.join(out_dir, f"preds_{run_id}_tta.parquet"), index=False)
    ap50, ap = coco_eval(test_ann, p_plain)
    ap50_t, ap_t = coco_eval(test_ann, p_tta)
    row = dict(run_id=run_id, recipe=name, seed=seed, best_epoch=best_epoch,
               val_ap50=best_val if best_val >= 0 else np.nan,
               test_ap50=ap50, test_ap50_95=ap, test_ap50_tta=ap50_t, test_ap50_95_tta=ap_t,
               train_minutes=(time.time() - t0) / 60, **{f"cfg_{k}": v for k, v in cfg.items()})
    print(f"  [{run_id}] TEST  AP50 {ap50:.2f}  AP50:95 {ap:.2f}  |  with TTA: AP50 {ap50_t:.2f}  "
          f"AP50:95 {ap_t:.2f}")
    return row


def append_result(row, path=RESULTS_CSV):
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def train_all():
    os.makedirs(OUT_DIR, exist_ok=True)
    done = set(pd.read_csv(RESULTS_CSV)["run_id"]) if os.path.exists(RESULTS_CSV) else set()
    start = time.time()
    durations = []
    for name, seed in RUN_PLAN:
        run_id = f"{name}_s{seed}"
        if run_id in done:
            print(f"skip {run_id} (already in results)")
            continue
        elapsed_h = (time.time() - start) / 3600
        est_h = (max(durations) if durations else 0) / 60
        if elapsed_h + est_h > TIME_BUDGET_HOURS:
            print(f"Stopping: {elapsed_h:.1f} h used, next run ~{est_h:.1f} h would exceed "
                  f"the {TIME_BUDGET_HOURS} h budget. Save {OUT_DIR} and re-run in a new session.")
            break
        print(f"\n=== {run_id} ===  config: {EXPERIMENTS[name]}")
        row = train_one(name, seed)
        append_result(row)
        durations.append(row["train_minutes"])
    print("\nDone for this session. Run MODE = 'report' for the comparison.")


# ----------------------------------------------------------------------
# Report
# ----------------------------------------------------------------------
def report(out_dir=OUT_DIR, ann_dir=ANN_DIR, make_chart=True, n_boot=None):
    n_boot = n_boot or N_BOOTSTRAP
    res = pd.read_csv(os.path.join(out_dir, "results.csv"))
    test_ann = f"{ann_dir}/instances_test.json"
    with open(test_ann) as f:
        img_ids = [im["id"] for im in json.load(f)["images"]]
    lines = []
    P = lambda s="": (print(s), lines.append(s))
    recipes = [r for r in EXPERIMENTS if r in set(res.recipe)]
    suffixes = ("", "_tta")

    # Per-image matches at every IoU threshold, for every run (plain + TTA)
    matches = {}
    for run_id in res.run_id:
        for sfx in suffixes:
            p = os.path.join(out_dir, f"preds_{run_id}{sfx}.parquet")
            if os.path.exists(p):
                pr = pd.read_parquet(p)
                for t in IOU_THRS:
                    matches[(run_id + sfx, t)] = per_image_matches(test_ann, pr, iou_thr=t)

    def run_ap(key, ids, metric):
        thrs = [0.5] if metric == "ap50" else IOU_THRS
        return np.mean([ap_from_matches(matches[(key, t)], ids) for t in thrs])

    def recipe_ap(recipe, sfx, ids, metric):
        keys = [r + sfx for r in res[res.recipe == recipe].run_id if (r + sfx, 0.5) in matches]
        return np.mean([run_ap(k, ids, metric) for k in keys])

    best = res.sort_values("test_ap50").iloc[-1]
    c50, c5095 = run_ap(best.run_id, img_ids, "ap50"), run_ap(best.run_id, img_ids, "ap5095")

    rng = np.random.default_rng(0)
    samples = [rng.choice(img_ids, size=len(img_ids), replace=True) for _ in range(n_boot)]
    boot = {(r, sfx, m): np.array([recipe_ap(r, sfx, s_, m) for s_ in samples])
            for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}
    ci = {k: np.nanpercentile(v, [2.5, 97.5]) for k, v in boot.items()}
    col = {("", "ap50"): "test_ap50", ("_tta", "ap50"): "test_ap50_tta",
           ("", "ap5095"): "test_ap50_95", ("_tta", "ap5095"): "test_ap50_95_tta"}
    mean = {(r, sfx, m): res.loc[res.recipe == r, col[(sfx, m)]].mean()
            for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}
    sd = {(r, sfx, m): res.loc[res.recipe == r, col[(sfx, m)]].std()
          for r in recipes for sfx in suffixes for m in ("ap50", "ap5095")}

    P("=" * 106)
    P("INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)")
    P("=" * 106)
    P("mean over seeds ± sd across seeds   [95% bootstrap range over test images]")
    P()
    P(f"{'':48s} {'mAP@0.5':>26s}   {'mAP@0.5:0.95':>26s}")
    for r in recipes:
        for sfx in suffixes:
            name = RECIPE_LABELS.get(r, r) + (" + TTA" if sfx else "")
            cells = []
            for m in ("ap50", "ap5095"):
                lo, hi = ci[(r, sfx, m)]
                s_ = sd[(r, sfx, m)]
                cells.append(f"{mean[(r, sfx, m)]:5.2f} ± {s_:4.2f} [{lo:5.1f}-{hi:5.1f}]"
                             if pd.notna(s_) else f"{mean[(r, sfx, m)]:5.2f}        [{lo:5.1f}-{hi:5.1f}]")
            P(f"{name:48s} {cells[0]:>26s}   {cells[1]:>26s}")
    P()
    P(f"Metric check on {best.run_id}: fast mAP@0.5 {c50:.2f} vs pycocotools {best.test_ap50:.2f}; "
      f"fast mAP@0.5:0.95 {c5095:.2f} vs pycocotools {best.test_ap50_95:.2f} (should agree within ~0.5)")
    P()
    P("Individual runs:")
    P(res[["run_id", "best_epoch", "val_ap50", "test_ap50", "test_ap50_95",
           "test_ap50_tta"]].round(2).to_string(index=False))

    if "baseline" in recipes:
        P()
        P("Better than our original training? (paired bootstrap of the difference;")
        P("'yes' only if the whole 95% range is above 0)")
        for m, mname in (("ap50", "mAP@0.5"), ("ap5095", "mAP@0.5:0.95")):
            P(f"  {mname}:")
            for r in recipes:
                for sfx in suffixes:
                    if r == "baseline" and sfx == "":
                        continue
                    d = boot[(r, sfx, m)] - boot[("baseline", "", m)]
                    lo, hi = np.nanpercentile(d, [2.5, 97.5])
                    v = "yes" if lo > 0 else ("no -- worse" if hi < 0 else "not distinguishable")
                    P(f"    {RECIPE_LABELS.get(r, r) + (' + TTA' if sfx else ''):40s} {np.nanmean(d):+6.2f} "
                      f"(95%: {lo:+.2f} to {hi:+.2f}) -> {v}")

    others = [r for r in recipes if r not in OUR_RECIPES]
    ours = [r for r in recipes if r in OUR_RECIPES]
    if ours and others:
        P()
        P("Our improved Faster R-CNN vs every other design, all trained and tested by us")
        P("on the SAME data and split (no TTA; paired bootstrap of the difference):")
        for m, mname in (("ap50", "mAP@0.5"), ("ap5095", "mAP@0.5:0.95")):
            P(f"  {mname}:")
            for o in ours:
                for c in others:
                    d = boot[(o, "", m)] - boot[(c, "", m)]
                    lo, hi = np.nanpercentile(d, [2.5, 97.5])
                    v = "better" if lo > 0 else ("worse" if hi < 0 else "not distinguishable")
                    P(f"    {RECIPE_LABELS.get(o, o):42s} vs {RECIPE_LABELS.get(c, c):40s} "
                      f"{np.nanmean(d):+6.2f} (95%: {lo:+.2f} to {hi:+.2f}) -> {v}")

    P()
    P("Published numbers -- CONTEXT ONLY, not compared:")
    P("  " + PUBLISHED_NOTE)
    for pname, p50, p5095 in PUBLISHED:
        P(f"  {pname:38s} mAP@0.5 {p50:6.2f}   mAP@0.5:0.95 " + (f"{p5095:6.2f}" if p5095 else "  n/r"))

    with open(os.path.join(out_dir, "report.txt"), "w") as f:
        f.write("\n".join(lines))
    if make_chart:
        _charts(out_dir, recipes, mean, ci)
        P(f"\nSaved report.txt, comparison_chart.png (detailed) and presentation_chart.png in {out_dir}")


def _charts(out_dir, recipes, mean, ci):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    OURS, OTHER, INK, INK2, SURF = "#2a78d6", "#b9b8b2", "#0b0b0b", "#52514e", "#fcfcfb"
    plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2,
                         "xtick.color": INK2, "ytick.color": INK})

    # 1) Detailed: every recipe, with and without TTA, mAP@0.5 only
    rows = []
    for r in recipes:
        for sfx in ("", "_tta"):
            rows.append((RECIPE_LABELS.get(r, r) + (" + TTA" if sfx else ""), mean[(r, sfx, "ap50")],
                         ci[(r, sfx, "ap50")], OURS if r in OUR_RECIPES else OTHER))
    fig, ax = plt.subplots(figsize=(9.5, 0.45 * len(rows) + 1.4), facecolor=SURF)
    _bars(ax, rows, "mAP@0.5 (%)", INK)
    ax.set_title("Internal-wave detection, S1-IW-2023 test set (all variants, trained by us)", loc="left", fontsize=11)
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, "comparison_chart.png"), dpi=130, facecolor=SURF)
    plt.close(fig)

    # 2) Presentation: one bar per model, both metrics, sorted by mAP@0.5
    entries = [(RECIPE_LABELS.get(r, r), mean[(r, "", "ap50")], ci[(r, "", "ap50")],
                mean[(r, "", "ap5095")], ci[(r, "", "ap5095")], OURS if r in OUR_RECIPES else OTHER)
               for r in recipes]
    entries.sort(key=lambda e: e[1], reverse=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.5 * len(entries) + 2.0), facecolor=SURF, sharey=True)
    _bars(axes[0], [(e[0], e[1], e[2], e[5]) for e in entries], "mAP@0.5 (%)", INK)
    _bars(axes[1], [(e[0], e[3], e[4], e[5]) for e in entries], "mAP@0.5:0.95 (%)  -- stricter: rewards tight boxes",
          INK, show_labels=False)
    axes[0].set_title("Detection (box overlaps the wave by at least 50%)", loc="left", fontsize=11)
    axes[1].set_title("Detection + precise box placement", loc="left", fontsize=11)
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(color=OURS, label="Our method: Faster R-CNN + improved training"),
                        Patch(color=OTHER, label="Other designs / original training")],
               loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle("Internal-wave detection on the S1-IW-2023 test set -- all models trained and tested "
                 "by us on the same data (mean of 3 runs; line = 95% range)", x=0.01, ha="left",
                 fontsize=12, fontweight="bold")
    fig.subplots_adjust(left=0.24, right=0.98, top=0.86, bottom=0.17, wspace=0.08)
    fig.savefig(os.path.join(out_dir, "presentation_chart.png"), dpi=150, facecolor=SURF)
    plt.close(fig)


def _bars(ax, rows, xlabel, ink, show_labels=True):
    """rows: (label, value or None, (lo, hi) or None, color). Top row first."""
    y = np.arange(len(rows))[::-1]
    vals = [r[1] for r in rows if r[1] is not None]
    lo_x = max(0, np.floor((min(vals) - 8) / 5) * 5)
    for yy, (lbl, v, rng_, c) in zip(y, rows):
        if v is None:
            ax.text(lo_x + 0.5, yy, "not reported", va="center", fontsize=9, color="#8a8984", style="italic")
            continue
        ax.barh(yy, v - lo_x, left=lo_x, height=0.62, color=c, edgecolor="none")
        if rng_ is not None:
            ax.plot(rng_, [yy, yy], color=ink, lw=1.2)
            ax.plot([rng_[0]] * 2, [yy - 0.13, yy + 0.13], color=ink, lw=1.2)
            ax.plot([rng_[1]] * 2, [yy - 0.13, yy + 0.13], color=ink, lw=1.2)
        x_txt = (rng_[1] if rng_ is not None else v) + 0.4
        ax.text(x_txt, yy, f"{v:.1f}", va="center", fontsize=9, color=ink)
    if show_labels:
        ax.set_yticks(y)
        ax.set_yticklabels([r[0] for r in rows])
    else:
        ax.tick_params(labelleft=False)
    ax.set_xlim(lo_x, 100)
    ax.set_ylim(-0.6, len(rows) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", color="#e4e3df", lw=0.8)
    ax.set_axisbelow(True)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(axis="y", length=0)


# ----------------------------------------------------------------------
# Self-test (CPU, synthetic, ~1-2 min)
# ----------------------------------------------------------------------
def selftest():
    import tempfile
    print("=" * 70 + "\nSELF-TEST\n" + "=" * 70)
    ok_all = True

    print("\n[1] Augmentation keeps boxes on the objects (all 8 orientations)")
    torch.manual_seed(0); random.seed(0)
    worst = 0.0
    for _ in range(64):
        img = torch.zeros(3, 120, 200)
        bx = torch.tensor([[30., 20., 90., 50.], [120., 60., 180., 110.]])
        for x1, y1, x2, y2 in bx.int().tolist():
            img[:, y1:y2, x1:x2] = 1.0
        a_img, a_b = augment(img, bx)
        m = a_img[0] > 0.5
        for x1, y1, x2, y2 in a_b.round().int().tolist():
            inside = m[y1:y2, x1:x2].float().mean().item()
            worst = max(worst, 1 - inside)
        if abs(m.sum().item() - sum(((b[2] - b[0]) * (b[3] - b[1])).item() for b in a_b)) > 1:
            worst = 1.0
    ok = worst < 0.01
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  worst box/object mismatch {worst:.3f}")

    print("\n[2] Fast mAP (used for the confidence ranges) matches pycocotools")
    tmp = tempfile.mkdtemp()
    rng = np.random.default_rng(1)
    images, anns, preds = [], [], []
    aid = 1
    for iid in range(1, 41):
        images.append(dict(id=iid, file_name=f"{iid}.png", width=256, height=256))
        for _ in range(rng.integers(1, 4)):
            x, y = rng.uniform(0, 180, 2); w, h = rng.uniform(20, 70, 2)
            anns.append(dict(id=aid, image_id=iid, category_id=1, bbox=[x, y, w, h], area=w * h, iscrowd=0))
            aid += 1
            if rng.random() < 0.8:
                j = rng.normal(0, 6, 4)
                preds.append(dict(image_id=iid, x1=x + j[0], y1=y + j[1], x2=x + w + j[2],
                                  y2=y + h + j[3], score=rng.uniform(0.3, 1)))
        for _ in range(rng.integers(0, 3)):
            x, y = rng.uniform(0, 200, 2)
            preds.append(dict(image_id=iid, x1=x, y1=y, x2=x + 40, y2=y + 40, score=rng.uniform(0, 0.8)))
    ann_path = os.path.join(tmp, "t.json")
    with open(ann_path, "w") as f:
        json.dump(dict(images=images, annotations=anns, categories=[dict(id=1, name="w")]), f)
    preds = pd.DataFrame(preds)
    ref, ref95 = coco_eval(ann_path, preds)
    ids = [im["id"] for im in images]
    fast = ap_from_matches(per_image_matches(ann_path, preds), ids)
    fast95 = np.mean([ap_from_matches(per_image_matches(ann_path, preds, iou_thr=t), ids) for t in IOU_THRS])
    ok = abs(ref - fast) < 0.5 and abs(ref95 - fast95) < 0.5
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  mAP@0.5: pycocotools {ref:.2f} vs fast {fast:.2f} | "
          f"mAP@0.5:0.95: pycocotools {ref95:.2f} vs fast {fast95:.2f}")

    print("\n[3] Test-time augmentation maps flipped boxes back correctly")
    class Fake(torch.nn.Module):
        def forward(self, xs):                      # 'detects' the bright square
            outs = []
            for x in xs:
                ys, xs_ = torch.where(x[0] > 0.5)
                b = torch.tensor([[xs_.min(), ys.min(), xs_.max() + 1, ys.max() + 1]], dtype=torch.float32)
                outs.append(dict(boxes=b, scores=torch.tensor([0.9])))
            return outs
    img = torch.zeros(3, 100, 160); img[:, 10:30, 20:70] = 1
    b, s = predict(Fake().eval(), img, tta=True)
    ok = len(b) == 1 and torch.allclose(b[0], torch.tensor([20., 10., 70., 30.]), atol=0.5) and abs(s.item() - 0.9) < 1e-4
    ok_all &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  fused box {b.tolist()} score {s.tolist()}")

    print("\n[4] Every detector design builds, trains one step and predicts")
    for arch in ("v1", "v2", "retinanet", "fcos", "ssd"):
        try:
            mdl = build_model(arch, 192, pretrained=False)
            mdl.train()
            tgt = [{"boxes": torch.tensor([[20., 30., 120., 90.]]), "labels": torch.tensor([1])}]
            loss = sum(mdl([torch.rand(3, 192, 192)], tgt).values())
            loss.backward()
            mdl.eval()
            with torch.no_grad():
                o = mdl([torch.rand(3, 192, 192)])[0]
            ok = set(o) >= {"boxes", "scores", "labels"} and torch.isfinite(loss).item()
        except Exception as e:
            print("   ", arch, repr(e)); ok = False
        ok_all &= ok
        print(f"  {'PASS' if ok else 'FAIL'}  {arch}")

    print("\n[5] End-to-end: tiny synthetic dataset, 2 recipes x 2 seeds, train -> report (CPU)")
    img_dir = os.path.join(tmp, "img"); os.makedirs(img_dir)
    ann_dir = os.path.join(tmp, "ann"); os.makedirs(ann_dir)
    rng = np.random.default_rng(2)
    for split, n in (("train", 12), ("val", 4), ("test", 6)):
        ims, ans = [], []
        for k in range(n):
            iid = {"train": 0, "val": 100, "test": 200}[split] + k
            arr = (rng.normal(0.35, 0.08, (192, 192)) * 255).clip(0, 255).astype(np.uint8)
            x, y = rng.integers(10, 110, 2); w, h = rng.integers(40, 70, 2)
            for c in range(0, w, 12):                   # bright stripes = a 'wave packet'
                arr[y:y + h, x + c:x + c + 5] = 230
            Image.fromarray(arr).save(os.path.join(img_dir, f"{iid}.png"))
            ims.append(dict(id=iid, file_name=f"{iid}.png", width=192, height=192))
            ans.append(dict(id=iid + 1, image_id=iid, category_id=1, bbox=[int(x), int(y), int(w), int(h)],
                            area=int(w * h), iscrowd=0))
        with open(os.path.join(ann_dir, f"instances_{split}.json"), "w") as f:
            json.dump(dict(images=ims, annotations=ans, categories=[dict(id=1, name="internal wave")]), f)
    out = os.path.join(tmp, "out"); os.makedirs(out)
    global NUM_WORKERS, BATCH_SIZE
    nw, bs = NUM_WORKERS, BATCH_SIZE
    NUM_WORKERS, BATCH_SIZE = 0, 2
    try:
        for name, seed in [("baseline", 0), ("improved", 0), ("retinanet", 0), ("ssd", 0),
                           ("baseline", 1), ("improved", 1), ("retinanet", 1), ("ssd", 1)]:
            row = train_one(name, seed, ann_dir=ann_dir, image_dir=img_dir, out_dir=out, pretrained=False,
                            cfg_override=dict(epochs=2, img_size=192), log_every=1000)
            append_result(row, os.path.join(out, "results.csv"))
        report(out_dir=out, ann_dir=ann_dir, make_chart=True, n_boot=200)
        ok = all(os.path.exists(os.path.join(out, f)) for f in ("comparison_chart.png", "presentation_chart.png"))
    except Exception as e:
        import traceback; traceback.print_exc()
        ok = False
    finally:
        NUM_WORKERS, BATCH_SIZE = nw, bs
    ok_all &= ok
    print(f"\n  {'PASS' if ok else 'FAIL'}  pipeline ran end to end (numbers above are meaningless --"
          f" untrained model, 2 epochs, synthetic data)")

    print("\n" + "=" * 70)
    print("ALL SELF-TESTS PASSED." if ok_all else "SELF-TEST FAILURES -- fix before training.")
    print("=" * 70)
    return ok_all


def main():
    in_nb = "ipykernel" in sys.modules
    mode = MODE
    if not in_nb and len(sys.argv) == 2 and sys.argv[1] in ("selftest", "train", "report"):
        mode = sys.argv[1]
    print(f"Mode: {mode} | device: {DEVICE}")
    if mode == "selftest":
        selftest()
    elif mode == "train":
        train_all()
    elif mode == "report":
        report()
    else:
        print("MODE must be 'selftest', 'train' or 'report'")


if __name__ == "__main__":
    main()

Mode: report | device: cuda
INTERNAL-WAVE DETECTOR -- S1-IW-2023 held-out test set (74 images)
mean over seeds ± sd across seeds   [95% bootstrap range over test images]

                                                                    mAP@0.5                 mAP@0.5:0.95
Faster R-CNN, original training                  85.02 ± 1.43 [ 79.8- 90.7]   57.75 ± 1.69 [ 52.7- 63.5]
Faster R-CNN, original training + TTA            88.63 ± 1.42 [ 83.9- 93.4]   62.08 ± 1.11 [ 57.8- 67.1]
Faster R-CNN, improved training (ours)           91.08 ± 0.66 [ 86.9- 95.0]   61.26 ± 1.85 [ 56.8- 66.2]
Faster R-CNN, improved training (ours) + TTA     90.38 ± 0.35 [ 86.3- 94.4]   62.92 ± 0.86 [ 58.3- 68.1]
Faster R-CNN v2, improved training (ours)        92.27 ± 1.13 [ 88.3- 96.1]   64.38 ± 2.14 [ 60.1- 69.1]
Faster R-CNN v2, improved training (ours) + TTA  91.94 ± 1.35 [ 88.1- 95.6]   67.73 ± 0.49 [ 63.3- 72.7]
RetinaNet (R50-FPN v2), same training            84.38 ± 1.77 [ 79.4- 89.8]   56.41 ± 2.28 [ 5